# 01_Dataset_Audit_and_Preparation

Notebook ini digunakan untuk:

1. Membaca Dataset A dan Dataset B.
2. Membuat manifest seluruh citra.
3. Menyeragamkan label kelas target.
4. Mendeteksi citra rusak.
5. Menghitung SHA-256 dan perceptual hash.
6. Mendeteksi duplicate.
7. Memeriksa distribusi kelas.
8. Menampilkan sampel citra untuk validasi label.

Notebook ini tidak melakukan pembagian train, validation, dan test.

In [ ]:
!pip install -q ImageHash

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 11.3 MB/s eta 0:00:00


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from pathlib import Path
from datetime import datetime
from collections import defaultdict
import hashlib
import json
import os
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image, ImageFile, UnidentifiedImageError
import imagehash

from tqdm.auto import tqdm
from IPython.display import display

warnings.filterwarnings("ignore")

# Jangan izinkan PIL membaca file citra yang terpotong.
ImageFile.LOAD_TRUNCATED_IMAGES = False

print("Library berhasil dimuat.")

Library berhasil dimuat.


In [ ]:
# ============================================================
# PATH PROYEK DAN DATASET
# ============================================================

from pathlib import Path

ROOT_DIR = Path(
    "/content/drive/MyDrive/eksperimen_baru"
)

BASELINE_V2_DIR = ROOT_DIR / "baseline_v2"
DATASET_DIR = ROOT_DIR / "dataset"

# Dataset A: Kaggle Paddy Disease Classification
DATASET_A_ROOT = (
    DATASET_DIR
    / "dataset A"
    / "paddy-disease-classification"
)

DATASET_A_IMAGES_DIR = (
    DATASET_A_ROOT
    / "train_images"
)

DATASET_A_LABEL_CSV = (
    DATASET_A_ROOT
    / "train.csv"
)

# Folder test_images sengaja tidak digunakan
DATASET_A_TEST_IMAGES_DIR = (
    DATASET_A_ROOT
    / "test_images"
)

# Dataset B
DATASET_B_DIR = (
    DATASET_DIR
    / "dataset B"
    / "archive (3)"
)

CONFIG_PATH = (
    BASELINE_V2_DIR
    / "config"
    / "config.json"
)

MANIFEST_DIR = BASELINE_V2_DIR / "manifests"
METRICS_DIR = BASELINE_V2_DIR / "metrics"
FIGURES_DIR = BASELINE_V2_DIR / "figures"
LOGS_DIR = BASELINE_V2_DIR / "logs"
DOCUMENTATION_DIR = (
    BASELINE_V2_DIR
    / "documentation"
)

required_directories = [
    ROOT_DIR,
    BASELINE_V2_DIR,
    DATASET_DIR,
    DATASET_A_ROOT,
    DATASET_A_IMAGES_DIR,
    DATASET_B_DIR,
    MANIFEST_DIR,
    METRICS_DIR,
    FIGURES_DIR,
    LOGS_DIR,
    DOCUMENTATION_DIR
]

for directory in required_directories:
    if not directory.exists():
        raise FileNotFoundError(
            f"Folder tidak ditemukan: {directory}"
        )

if not DATASET_A_LABEL_CSV.exists():
    raise FileNotFoundError(
        f"train.csv tidak ditemukan: "
        f"{DATASET_A_LABEL_CSV}"
    )

if not CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"config.json tidak ditemukan: "
        f"{CONFIG_PATH}"
    )

print("ROOT DIR             :", ROOT_DIR)
print("BASELINE V2          :", BASELINE_V2_DIR)
print("DATASET A ROOT       :", DATASET_A_ROOT)
print("DATASET A IMAGES     :", DATASET_A_IMAGES_DIR)
print("DATASET A LABEL CSV  :", DATASET_A_LABEL_CSV)
print("DATASET A TEST       :", DATASET_A_TEST_IMAGES_DIR)
print("DATASET B            :", DATASET_B_DIR)
print("CONFIG               :", CONFIG_PATH)

ROOT DIR             : /content/drive/MyDrive/eksperimen_baru
BASELINE V2          : /content/drive/MyDrive/eksperimen_baru/baseline_v2
DATASET A ROOT       : /content/drive/MyDrive/eksperimen_baru/dataset/dataset A/paddy-disease-classification
DATASET A IMAGES     : /content/drive/MyDrive/eksperimen_baru/dataset/dataset A/paddy-disease-classification/train_images
DATASET A LABEL CSV  : /content/drive/MyDrive/eksperimen_baru/dataset/dataset A/paddy-disease-classification/train.csv
DATASET A TEST       : /content/drive/MyDrive/eksperimen_baru/dataset/dataset A/paddy-disease-classification/test_images
DATASET B            : /content/drive/MyDrive/eksperimen_baru/dataset/dataset B/archive (3)
CONFIG               : /content/drive/MyDrive/eksperimen_baru/baseline_v2/config/config.json


In [ ]:
with open(
    CONFIG_PATH,
    "r",
    encoding="utf-8"
) as file:
    CONFIG = json.load(file)

RANDOM_STATE = CONFIG["random_state"]
TARGET_CLASSES = CONFIG["target_classes"]

print("Random state  :", RANDOM_STATE)
print("Target classes:", TARGET_CLASSES)

Random state  : 42
Target classes: ['bacterial_leaf_blight', 'blast', 'tungro']


In [ ]:
IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp",
    ".tif",
    ".tiff"
}

def count_images(directory):
    directory = Path(directory)

    return sum(
        1
        for path in directory.rglob("*")
        if path.is_file()
        and path.suffix.lower() in IMAGE_EXTENSIONS
    )

def show_directory_structure(
    root_directory,
    maximum_depth=2,
    maximum_items=40
):
    root_directory = Path(root_directory)

    print(f"\nStruktur: {root_directory}")
    print("=" * 70)

    displayed_items = 0

    for path in sorted(
        root_directory.rglob("*")
    ):
        relative_path = path.relative_to(
            root_directory
        )

        depth = len(relative_path.parts)

        if depth > maximum_depth:
            continue

        indentation = "    " * (
            depth - 1
        )

        marker = (
            "[DIR]"
            if path.is_dir()
            else "[FILE]"
        )

        print(
            f"{indentation}"
            f"{marker} {path.name}"
        )

        displayed_items += 1

        if displayed_items >= maximum_items:
            print("... tampilan dipotong ...")
            break

show_directory_structure(
    DATASET_A_ROOT,
    maximum_depth=2
)

show_directory_structure(
    DATASET_B_DIR,
    maximum_depth=2
)

print("\nJumlah citra:")
print(
    "Dataset A train_images :",
    count_images(DATASET_A_IMAGES_DIR)
)

print(
    "Dataset A test_images  :",
    count_images(DATASET_A_TEST_IMAGES_DIR)
)

print(
    "Dataset B              :",
    count_images(DATASET_B_DIR)
)


Struktur: /content/drive/MyDrive/eksperimen_baru/dataset/dataset A/paddy-disease-classification
[FILE] sample_submission.csv
[DIR] test_images
    [FILE] 200001.jpg
    [FILE] 200002.jpg
    [FILE] 200003.jpg
    [FILE] 200004.jpg
    [FILE] 200005.jpg
    [FILE] 200006.jpg
    [FILE] 200007.jpg
    [FILE] 200008.jpg
    [FILE] 200009.jpg
    [FILE] 200010.jpg
    [FILE] 200011.jpg
    [FILE] 200012.jpg
    [FILE] 200013.jpg
    [FILE] 200014.jpg
    [FILE] 200015.jpg
    [FILE] 200016.jpg
    [FILE] 200017.jpg
    [FILE] 200018.jpg
    [FILE] 200019.jpg
    [FILE] 200020.jpg
    [FILE] 200021.jpg
    [FILE] 200022.jpg
    [FILE] 200023.jpg
    [FILE] 200024.jpg
    [FILE] 200025.jpg
    [FILE] 200026.jpg
    [FILE] 200027.jpg
    [FILE] 200028.jpg
    [FILE] 200029.jpg
    [FILE] 200030.jpg
    [FILE] 200031.jpg
    [FILE] 200032.jpg
    [FILE] 200033.jpg
    [FILE] 200034.jpg
    [FILE] 200035.jpg
    [FILE] 200036.jpg
    [FILE] 200037.jpg
    [FILE] 200038.jpg
... tampilan dipoton

In [ ]:
# ============================================================
# PEMERIKSAAN TRAIN.CSV DATASET A
# ============================================================

train_labels_dataframe = pd.read_csv(
    DATASET_A_LABEL_CSV
)

print(
    "Ukuran train.csv:",
    train_labels_dataframe.shape
)

print(
    "\nNama kolom:",
    train_labels_dataframe.columns.tolist()
)

print("\nLima baris pertama:")
display(
    train_labels_dataframe.head()
)

print("\nTipe data:")
display(
    train_labels_dataframe.dtypes
)

Ukuran train.csv: (10407, 4)

Nama kolom: ['image_id', 'label', 'variety', 'age']

Lima baris pertama:


,image_id,label,variety,age
0,100330.jpg,bacterial_leaf_blight,ADT45,45
1,100365.jpg,bacterial_leaf_blight,ADT45,45
2,100382.jpg,bacterial_leaf_blight,ADT45,45
3,100632.jpg,bacterial_leaf_blight,ADT45,45
4,101918.jpg,bacterial_leaf_blight,ADT45,45



Tipe data:


,0
image_id,object
label,object
variety,object
age,int64


In [ ]:
possible_label_columns = [
    "label",
    "class",
    "disease",
    "category",
    "target"
]

detected_label_column = None

for candidate in possible_label_columns:
    if candidate in train_labels_dataframe.columns:
        detected_label_column = candidate
        break

if detected_label_column is None:
    raise ValueError(
        "Kolom label tidak ditemukan. "
        "Periksa nama kolom train.csv."
    )

print(
    "Kolom label terdeteksi:",
    detected_label_column
)

label_distribution_all = (
    train_labels_dataframe[
        detected_label_column
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis("raw_label")
    .reset_index(name="jumlah")
)

display(label_distribution_all)

Kolom label terdeteksi: label


,raw_label,jumlah
0,normal,1764
1,blast,1738
2,hispa,1594
3,dead_heart,1442
4,tungro,1088
5,brown_spot,965
6,downy_mildew,620
7,bacterial_leaf_blight,479
8,bacterial_leaf_streak,380
9,bacterial_panicle_blight,337


In [ ]:
# ============================================================
# NORMALISASI DAN LABEL MAPPING
# ============================================================

import re


def normalize_text(value):
    value = str(value).strip().lower()

    return re.sub(
        pattern=r"[^a-z0-9]+",
        repl="",
        string=value
    )


DATASET_A_LABEL_MAPPING = {
    normalize_text(
        "bacterial_leaf_blight"
    ): "bacterial_leaf_blight",

    normalize_text(
        "bacterial leaf blight"
    ): "bacterial_leaf_blight",

    normalize_text(
        "bacterial_blight"
    ): "bacterial_leaf_blight",

    normalize_text("blast"): "blast",
    normalize_text("leaf_blast"): "blast",
    normalize_text("rice_blast"): "blast",

    normalize_text("tungro"): "tungro",
    normalize_text(
        "rice_tungro"
    ): "tungro"
}


DATASET_B_LABEL_MAPPING = {
    normalize_text(
        "bacterial_leaf_blight"
    ): "bacterial_leaf_blight",

    normalize_text(
        "bacterial leaf blight"
    ): "bacterial_leaf_blight",

    normalize_text(
        "blight"
    ): "bacterial_leaf_blight",

    normalize_text(
        "leaf_blight"
    ): "bacterial_leaf_blight",

    normalize_text("blast"): "blast",
    normalize_text("leaf_blast"): "blast",

    normalize_text("tungro"): "tungro",
    normalize_text(
        "rice_tungro"
    ): "tungro"
}


DATASET_B_REVIEW_LABELS = {
    normalize_text("blight"),
    normalize_text("leaf_blight")
}

print("Label mapping Dataset A:")
display(
    pd.DataFrame(
        DATASET_A_LABEL_MAPPING.items(),
        columns=[
            "normalized_raw_label",
            "standardized_label"
        ]
    )
)

print("\nLabel mapping Dataset B:")
display(
    pd.DataFrame(
        DATASET_B_LABEL_MAPPING.items(),
        columns=[
            "normalized_raw_label",
            "standardized_label"
        ]
    )
)

Label mapping Dataset A:


,normalized_raw_label,standardized_label
0,bacterialleafblight,bacterial_leaf_blight
1,bacterialblight,bacterial_leaf_blight
2,blast,blast
3,leafblast,blast
4,riceblast,blast
5,tungro,tungro
6,ricetungro,tungro



Label mapping Dataset B:


,normalized_raw_label,standardized_label
0,bacterialleafblight,bacterial_leaf_blight
1,blight,bacterial_leaf_blight
2,leafblight,bacterial_leaf_blight
3,blast,blast
4,leafblast,blast
5,tungro,tungro
6,ricetungro,tungro


In [ ]:
# ============================================================
# FUNGSI DETEKSI KOLOM CSV
# ============================================================

CSV_LABEL_COLUMN_CANDIDATES = [
    "label",
    "class",
    "disease",
    "category",
    "target"
]

CSV_IMAGE_COLUMN_CANDIDATES = [
    "image_id",
    "image",
    "filename",
    "file",
    "img",
    "path",
    "id"
]


def find_column(
    dataframe,
    column_candidates
):
    lowercase_columns = {
        str(column).strip().lower(): column
        for column in dataframe.columns
    }

    for candidate in column_candidates:
        if candidate in lowercase_columns:
            return lowercase_columns[candidate]

    return None

In [ ]:
# ============================================================
# MANIFEST DATASET A DARI TRAIN.CSV
# ============================================================

def list_image_paths(root_directory):
    root_directory = Path(root_directory)

    return sorted([
        path
        for path in root_directory.rglob("*")
        if path.is_file()
        and path.suffix.lower() in IMAGE_EXTENSIONS
    ])


def build_dataset_a_manifest(
    image_directory,
    label_csv_path,
    label_mapping
):
    image_directory = Path(
        image_directory
    )

    label_csv_path = Path(
        label_csv_path
    )

    labels_dataframe = pd.read_csv(
        label_csv_path
    )

    image_column = find_column(
        labels_dataframe,
        CSV_IMAGE_COLUMN_CANDIDATES
    )

    label_column = find_column(
        labels_dataframe,
        CSV_LABEL_COLUMN_CANDIDATES
    )

    if image_column is None:
        raise ValueError(
            "Kolom nama file citra "
            "tidak ditemukan di train.csv."
        )

    if label_column is None:
        raise ValueError(
            "Kolom label tidak ditemukan "
            "di train.csv."
        )

    print("Kolom citra:", image_column)
    print("Kolom label:", label_column)

    image_paths = list_image_paths(
        image_directory
    )

    # Indeks nama file dan stem untuk pencarian
    image_by_filename = {
        path.name.lower(): path
        for path in image_paths
    }

    image_by_stem = {}

    for path in image_paths:
        stem = path.stem.lower()

        if stem not in image_by_stem:
            image_by_stem[stem] = []

        image_by_stem[stem].append(path)

    records = []

    for _, row in labels_dataframe.iterrows():
        image_value = str(
            row[image_column]
        ).strip()

        raw_label = str(
            row[label_column]
        ).strip()

        image_name = Path(
            image_value
        ).name

        image_stem = Path(
            image_value
        ).stem.lower()

        resolved_path = image_by_filename.get(
            image_name.lower()
        )

        resolution_method = (
            "exact_filename"
            if resolved_path is not None
            else None
        )

        if resolved_path is None:
            stem_matches = image_by_stem.get(
                image_stem,
                []
            )

            if len(stem_matches) == 1:
                resolved_path = stem_matches[0]
                resolution_method = (
                    "unique_stem"
                )

        if resolved_path is None:
            # Digunakan untuk mencatat file yang hilang
            resolved_path = (
                image_directory
                / image_name
            )

            resolution_method = "not_found"

        normalized_raw_label = (
            normalize_text(raw_label)
        )

        standardized_label = (
            label_mapping.get(
                normalized_raw_label
            )
        )

        if standardized_label in TARGET_CLASSES:
            inclusion_status = "target_class"
        else:
            inclusion_status = (
                "excluded_non_target"
            )

        try:
            relative_path = str(
                resolved_path.relative_to(
                    image_directory
                )
            )
        except ValueError:
            relative_path = (
                resolved_path.name
            )

        records.append({
            "dataset": "Dataset_A",
            "path": str(resolved_path),
            "relative_path": relative_path,
            "filename": resolved_path.name,
            "extension": (
                resolved_path.suffix.lower()
            ),
            "raw_label": raw_label,
            "normalized_raw_label": (
                normalized_raw_label
            ),
            "label": standardized_label,
            "label_source": "train.csv",
            "label_reference": str(
                label_csv_path
            ),
            "inclusion_status": (
                inclusion_status
            ),
            "manual_label_review_required": (
                False
            ),
            "file_resolution_method": (
                resolution_method
            ),
            "file_exists": (
                resolved_path.exists()
            )
        })

    manifest = pd.DataFrame(records)

    print(
        "Jumlah baris train.csv:",
        len(labels_dataframe)
    )

    print(
        "Jumlah gambar train_images:",
        len(image_paths)
    )

    print(
        "Jumlah file tidak ditemukan:",
        int(
            (~manifest["file_exists"]).sum()
        )
    )

    return manifest

In [ ]:
# ============================================================
# MANIFEST DATASET B DARI STRUKTUR FOLDER
# ============================================================

def infer_label_from_directory(
    image_path,
    root_directory,
    label_mapping
):
    image_path = Path(image_path)
    root_directory = Path(
        root_directory
    )

    relative_parts = image_path.relative_to(
        root_directory
    ).parts[:-1]

    for folder_name in reversed(
        relative_parts
    ):
        normalized_folder = (
            normalize_text(folder_name)
        )

        if normalized_folder in label_mapping:
            return (
                folder_name,
                "directory_alias"
            )

    return (
        image_path.parent.name,
        "parent_directory"
    )


def build_dataset_b_manifest(
    root_directory,
    label_mapping,
    review_labels
):
    root_directory = Path(
        root_directory
    )

    image_paths = list_image_paths(
        root_directory
    )

    records = []

    for image_path in image_paths:
        (
            raw_label,
            label_source
        ) = infer_label_from_directory(
            image_path=image_path,
            root_directory=root_directory,
            label_mapping=label_mapping
        )

        normalized_raw_label = (
            normalize_text(raw_label)
        )

        standardized_label = (
            label_mapping.get(
                normalized_raw_label
            )
        )

        if standardized_label in TARGET_CLASSES:
            inclusion_status = "target_class"
        else:
            inclusion_status = (
                "excluded_non_target"
            )

        records.append({
            "dataset": "Dataset_B",
            "path": str(image_path),
            "relative_path": str(
                image_path.relative_to(
                    root_directory
                )
            ),
            "filename": image_path.name,
            "extension": (
                image_path.suffix.lower()
            ),
            "raw_label": raw_label,
            "normalized_raw_label": (
                normalized_raw_label
            ),
            "label": standardized_label,
            "label_source": label_source,
            "label_reference": str(
                image_path.parent
            ),
            "inclusion_status": (
                inclusion_status
            ),
            "manual_label_review_required": (
                normalized_raw_label
                in review_labels
            ),
            "file_resolution_method": (
                "directory_scan"
            ),
            "file_exists": True
        })

    return pd.DataFrame(records)

In [ ]:
from pathlib import Path
import pandas as pd

# ============================================================
# FUNGSI DETEKSI KOLOM CSV (Moved from 0cvkCh4MEh4R)
# ============================================================

CSV_LABEL_COLUMN_CANDIDATES = [
    "label",
    "class",
    "disease",
    "category",
    "target"
]

CSV_IMAGE_COLUMN_CANDIDATES = [
    "image_id",
    "image",
    "filename",
    "file",
    "img",
    "path",
    "id"
]


def find_column(
    dataframe,
    column_candidates
):
    lowercase_columns = {
        str(column).strip().lower(): column
        for column in dataframe.columns
    }

    for candidate in column_candidates:
        if candidate in lowercase_columns:
            return lowercase_columns[candidate]

    return None


# ============================================================
# MEMBANGUN MANIFEST DATASET A DAN DATASET B
# ============================================================

dataset_a_all = build_dataset_a_manifest(
    image_directory=DATASET_A_IMAGES_DIR,
    label_csv_path=DATASET_A_LABEL_CSV,
    label_mapping=DATASET_A_LABEL_MAPPING
)

dataset_b_all = build_dataset_b_manifest(
    root_directory=DATASET_B_DIR,
    label_mapping=DATASET_B_LABEL_MAPPING,
    review_labels=DATASET_B_REVIEW_LABELS
)

print("\nDataset A — status inklusi:")
display(
    dataset_a_all[
        "inclusion_status"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis("status")
    .reset_index(name="jumlah")
)

print("\nDataset B — status inklusi:")
display(
    dataset_b_all[
        "inclusion_status"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis("status")
    .reset_index(name="jumlah")
)

print("\nDataset A — resolusi file:")
display(
    dataset_a_all[
        "file_resolution_method"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "resolution_method"
    )
    .reset_index(name="jumlah")
)

Kolom citra: image_id
Kolom label: label
Jumlah baris train.csv: 10407
Jumlah gambar train_images: 10407
Jumlah file tidak ditemukan: 0

Dataset A — status inklusi:


,status,jumlah
0,excluded_non_target,7102
1,target_class,3305



Dataset B — status inklusi:


,status,jumlah
0,target_class,240



Dataset A — resolusi file:


,resolution_method,jumlah
0,exact_filename,10407


In [ ]:
# ============================================================
# DIAGNOSIS DISTRIBUSI LABEL DATASET A
# ============================================================

raw_label_distribution = (
    dataset_a_all
    .groupby(
        [
            "raw_label",
            "label",
            "inclusion_status"
        ],
        dropna=False
    )
    .size()
    .reset_index(name="jumlah")
    .sort_values(
        "jumlah",
        ascending=False
    )
    .reset_index(drop=True)
)

display(raw_label_distribution)

,raw_label,label,inclusion_status,jumlah
0,normal,NaN,excluded_non_target,1764
1,blast,blast,target_class,1738
2,hispa,NaN,excluded_non_target,1594
3,dead_heart,NaN,excluded_non_target,1442
4,tungro,tungro,target_class,1088
5,brown_spot,NaN,excluded_non_target,965
6,downy_mildew,NaN,excluded_non_target,620
7,bacterial_leaf_blight,bacterial_leaf_blight,target_class,479
8,bacterial_leaf_streak,NaN,excluded_non_target,380
9,bacterial_panicle_blight,NaN,excluded_non_target,337


In [ ]:
# ============================================================
# MENYIMPAN DISTRIBUSI LABEL ASLI DATASET A
# ============================================================

RAW_LABEL_DISTRIBUTION_PATH = (
    METRICS_DIR
    / "dataset_a_raw_label_distribution.csv"
)

raw_label_distribution.to_csv(
    RAW_LABEL_DISTRIBUTION_PATH,
    index=False
)

print(
    "Distribusi label asli Dataset A disimpan di:"
)
print(RAW_LABEL_DISTRIBUTION_PATH)

Distribusi label asli Dataset A disimpan di:
/content/drive/MyDrive/eksperimen_baru/baseline_v2/metrics/dataset_a_raw_label_distribution.csv


In [ ]:
# ============================================================
# DIAGNOSIS LABEL YANG MENGANDUNG KATA BLIGHT
# ============================================================

blight_label_diagnosis = (
    dataset_a_all[
        dataset_a_all[
            "raw_label"
        ]
        .astype(str)
        .str.contains(
            "blight",
            case=False,
            na=False
        )
    ]
    .groupby(
        [
            "raw_label",
            "label",
            "inclusion_status"
        ],
        dropna=False
    )
    .size()
    .reset_index(name="jumlah")
)

BLIGHT_DIAGNOSIS_PATH = (
    METRICS_DIR
    / "dataset_a_blight_label_diagnosis.csv"
)

blight_label_diagnosis.to_csv(
    BLIGHT_DIAGNOSIS_PATH,
    index=False
)

display(blight_label_diagnosis)

print(
    "\nDiagnosis label blight disimpan di:"
)
print(BLIGHT_DIAGNOSIS_PATH)

,raw_label,label,inclusion_status,jumlah
0,bacterial_leaf_blight,bacterial_leaf_blight,target_class,479
1,bacterial_panicle_blight,NaN,excluded_non_target,337



Diagnosis label blight disimpan di:
/content/drive/MyDrive/eksperimen_baru/baseline_v2/metrics/dataset_a_blight_label_diagnosis.csv


In [ ]:
target_distribution = (
    dataset_a_all[
        dataset_a_all[
            "inclusion_status"
        ] == "target_class"
    ]
    ["label"]
    .value_counts()
    .reindex(
        TARGET_CLASSES,
        fill_value=0
    )
    .rename_axis("label")
    .reset_index(name="jumlah")
)

display(target_distribution)

print(
    "Total kelas target:",
    target_distribution["jumlah"].sum()
)

,label,jumlah
0,bacterial_leaf_blight,479
1,blast,1738
2,tungro,1088


Total kelas target: 3305


In [ ]:
# ============================================================
# CELL 12 — MEMILIH TIGA KELAS TARGET
# ============================================================

dataset_a_target = (
    dataset_a_all[
        dataset_a_all["inclusion_status"] == "target_class"
    ]
    .copy()
    .reset_index(drop=True)
)

dataset_b_target = (
    dataset_b_all[
        dataset_b_all["inclusion_status"] == "target_class"
    ]
    .copy()
    .reset_index(drop=True)
)

combined_target = pd.concat(
    [
        dataset_a_target,
        dataset_b_target
    ],
    ignore_index=True
)

print("Jumlah data target sebelum audit citra:")
print("Dataset A :", len(dataset_a_target))
print("Dataset B :", len(dataset_b_target))
print("Gabungan  :", len(combined_target))

print("\nDistribusi kelas Dataset A:")
display(
    dataset_a_target["label"]
    .value_counts()
    .reindex(TARGET_CLASSES, fill_value=0)
    .rename_axis("label")
    .reset_index(name="jumlah")
)

print("\nDistribusi kelas Dataset B:")
display(
    dataset_b_target["label"]
    .value_counts()
    .reindex(TARGET_CLASSES, fill_value=0)
    .rename_axis("label")
    .reset_index(name="jumlah")
)

Jumlah data target sebelum audit citra:
Dataset A : 3305
Dataset B : 240
Gabungan  : 3545

Distribusi kelas Dataset A:


,label,jumlah
0,bacterial_leaf_blight,479
1,blast,1738
2,tungro,1088



Distribusi kelas Dataset B:


,label,jumlah
0,bacterial_leaf_blight,80
1,blast,80
2,tungro,80


In [ ]:
# ============================================================
# CELL 13 — VALIDASI MANIFEST TARGET
# ============================================================

def validate_target_manifest(
    dataframe,
    dataset_name
):
    available_classes = set(
        dataframe["label"].dropna().unique()
    )

    missing_classes = (
        set(TARGET_CLASSES)
        - available_classes
    )

    duplicated_paths = int(
        dataframe["path"].duplicated().sum()
    )

    missing_files = int(
        (~dataframe["file_exists"]).sum()
    )

    unresolved_labels = int(
        dataframe["label"].isna().sum()
    )

    print(f"\nValidasi {dataset_name}")
    print("-" * 50)
    print("Kelas tersedia      :", sorted(available_classes))
    print("Kelas tidak tersedia:", sorted(missing_classes))
    print("Path duplikat        :", duplicated_paths)
    print("File tidak ditemukan :", missing_files)
    print("Label kosong         :", unresolved_labels)

    if missing_classes:
        raise ValueError(
            f"{dataset_name} kehilangan kelas: {missing_classes}"
        )

    if duplicated_paths > 0:
        raise ValueError(
            f"{dataset_name} memiliki {duplicated_paths} path duplikat."
        )

    if missing_files > 0:
        raise FileNotFoundError(
            f"{dataset_name} memiliki {missing_files} file tidak ditemukan."
        )

    if unresolved_labels > 0:
        raise ValueError(
            f"{dataset_name} memiliki {unresolved_labels} label kosong."
        )


validate_target_manifest(
    dataset_a_target,
    "Dataset A"
)

validate_target_manifest(
    dataset_b_target,
    "Dataset B"
)

print("\nValidasi manifest target: LULUS")


Validasi Dataset A
--------------------------------------------------
Kelas tersedia      : ['bacterial_leaf_blight', 'blast', 'tungro']
Kelas tidak tersedia: []
Path duplikat        : 0
File tidak ditemukan : 0
Label kosong         : 0

Validasi Dataset B
--------------------------------------------------
Kelas tersedia      : ['bacterial_leaf_blight', 'blast', 'tungro']
Kelas tidak tersedia: []
Path duplikat        : 0
File tidak ditemukan : 0
Label kosong         : 0

Validasi manifest target: LULUS


In [ ]:
# ============================================================
# CELL 14 — FUNGSI AUDIT CITRA
# ============================================================

from pathlib import Path
import hashlib

import pandas as pd
from PIL import Image, ImageFile, UnidentifiedImageError
from tqdm.auto import tqdm
import imagehash

# File citra yang terpotong dianggap bermasalah.
ImageFile.LOAD_TRUNCATED_IMAGES = False


def calculate_sha256(
    file_path,
    chunk_size=1024 * 1024
):
    """
    Menghitung SHA-256 file untuk mendeteksi
    exact duplicate.
    """
    sha256_hash = hashlib.sha256()

    with open(file_path, "rb") as file:
        while True:
            chunk = file.read(chunk_size)

            if not chunk:
                break

            sha256_hash.update(chunk)

    return sha256_hash.hexdigest()


def audit_image_file(file_path):
    """
    Membaca satu file citra dan menghasilkan
    informasi teknis serta nilai hash.
    """
    file_path = Path(file_path)

    result = {
        "path": str(file_path),
        "file_size_bytes": None,
        "width": None,
        "height": None,
        "aspect_ratio": None,
        "color_mode": None,
        "image_format": None,
        "sha256": None,
        "phash": None,
        "audit_status": "valid",
        "audit_error": None
    }

    try:
        if not file_path.exists():
            raise FileNotFoundError(
                f"File tidak ditemukan: {file_path}"
            )

        result["file_size_bytes"] = int(
            file_path.stat().st_size
        )

        result["sha256"] = calculate_sha256(
            file_path
        )

        with Image.open(file_path) as image:
            # Memaksa seluruh isi file citra dibaca.
            image.load()

            width = int(image.width)
            height = int(image.height)

            result["width"] = width
            result["height"] = height

            if height > 0:
                result["aspect_ratio"] = (
                    width / height
                )

            result["color_mode"] = str(image.mode)
            result["image_format"] = str(
                image.format
            )

            rgb_image = image.convert("RGB")

            result["phash"] = str(
                imagehash.phash(
                    rgb_image,
                    hash_size=8
                )
            )

    except (
        FileNotFoundError,
        UnidentifiedImageError,
        OSError,
        ValueError,
        SyntaxError
    ) as error:
        result["audit_status"] = "corrupt"
        result["audit_error"] = (
            f"{type(error).__name__}: {error}"
        )

    except Exception as error:
        result["audit_status"] = "error"
        result["audit_error"] = (
            f"{type(error).__name__}: {error}"
        )

    return result


print("Fungsi audit citra berhasil dibuat.")

Fungsi audit citra berhasil dibuat.


In [ ]:
# ============================================================
# CELL 15 — MENJALANKAN AUDIT SELURUH CITRA
# RESTART-SAFE / RESUMABLE
# ============================================================

AUDIT_RAW_PATH = (
    LOGS_DIR
    / "image_audit_raw.csv"
)

CHECKPOINT_EVERY = 100


image_paths = (
    combined_target[
        "path"
    ]
    .astype(str)
    .tolist()
)

expected_paths = set(
    image_paths
)


print(
    "Jumlah citra yang harus diaudit:",
    len(image_paths)
)


# ------------------------------------------------------------
# 1. LOAD CHECKPOINT JIKA ADA
# ------------------------------------------------------------

if AUDIT_RAW_PATH.exists():

    previous_audit = pd.read_csv(
        AUDIT_RAW_PATH
    )

    if (
        not previous_audit.empty
        and
        "path" not in previous_audit.columns
    ):
        raise RuntimeError(
            "Checkpoint audit lama tidak valid: "
            "kolom 'path' tidak ditemukan."
        )


    if "path" in previous_audit.columns:

        previous_audit[
            "path"
        ] = (
            previous_audit[
                "path"
            ]
            .astype(str)
        )


        # Hanya mempertahankan path yang masih
        # terdapat pada manifest sekarang.
        previous_audit = (
            previous_audit[
                previous_audit[
                    "path"
                ].isin(
                    expected_paths
                )
            ]
            .drop_duplicates(
                subset=["path"],
                keep="last"
            )
            .reset_index(drop=True)
        )

    else:

        previous_audit = pd.DataFrame()


    completed_paths = set(
        previous_audit.get(
            "path",
            pd.Series(
                dtype=str
            )
        )
        .astype(str)
        .tolist()
    )


    audit_records = (
        previous_audit
        .to_dict(
            "records"
        )
    )


    print(
        "Checkpoint ditemukan:",
        len(completed_paths),
        "citra sudah selesai"
    )

else:

    completed_paths = set()

    audit_records = []

    print(
        "Checkpoint belum tersedia. "
        "Audit dimulai dari awal."
    )


# ------------------------------------------------------------
# 2. IDENTIFIKASI CITRA YANG BELUM DIAUDIT
# ------------------------------------------------------------

remaining_paths = [
    path
    for path in image_paths
    if path not in completed_paths
]


print(
    "Citra yang belum diaudit:",
    len(remaining_paths)
)


# ------------------------------------------------------------
# 3. JALANKAN AUDIT
# ------------------------------------------------------------

records_since_checkpoint = 0


for file_path in tqdm(
    remaining_paths,
    desc="Mengaudit citra",
    unit="citra"
):

    audit_result = (
        audit_image_file(
            file_path
        )
    )

    audit_records.append(
        audit_result
    )

    records_since_checkpoint += 1


    # --------------------------------------------------------
    # Periodic checkpoint
    # --------------------------------------------------------

    if (
        records_since_checkpoint
        >= CHECKPOINT_EVERY
    ):

        checkpoint_dataframe = (
            pd.DataFrame(
                audit_records
            )
            .drop_duplicates(
                subset=["path"],
                keep="last"
            )
            .reset_index(drop=True)
        )


        checkpoint_dataframe.to_csv(
            AUDIT_RAW_PATH,
            index=False
        )


        records_since_checkpoint = 0


# ------------------------------------------------------------
# 4. FINAL AUDIT DATAFRAME
# ------------------------------------------------------------

audit_dataframe = (
    pd.DataFrame(
        audit_records
    )
    .drop_duplicates(
        subset=["path"],
        keep="last"
    )
    .reset_index(drop=True)
)


audit_dataframe.to_csv(
    AUDIT_RAW_PATH,
    index=False
)


# ------------------------------------------------------------
# 5. HARD COMPLETION CHECK
# ------------------------------------------------------------

audited_paths = set(
    audit_dataframe[
        "path"
    ]
    .astype(str)
    .tolist()
)


missing_audit_paths = (
    expected_paths
    -
    audited_paths
)


unexpected_audit_paths = (
    audited_paths
    -
    expected_paths
)


if missing_audit_paths:

    raise RuntimeError(
        "AUDIT BELUM SELESAI.\n"
        f"{len(missing_audit_paths)} citra "
        "belum memiliki hasil audit."
    )


if unexpected_audit_paths:

    raise RuntimeError(
        "Audit dataframe berisi path "
        "yang bukan bagian manifest saat ini.\n"
        f"Jumlah: {len(unexpected_audit_paths)}"
    )


if (
    len(audit_dataframe)
    !=
    len(combined_target)
):

    raise RuntimeError(
        "Jumlah hasil audit tidak sama "
        "dengan jumlah manifest target.\n"
        f"Audit    : {len(audit_dataframe)}\n"
        f"Expected : {len(combined_target)}"
    )


if (
    audit_dataframe[
        "path"
    ]
    .duplicated()
    .any()
):

    raise RuntimeError(
        "Terdapat duplicate path "
        "pada audit_dataframe."
    )


# ------------------------------------------------------------
# 6. MERGE DENGAN MANIFEST
# ------------------------------------------------------------

combined_audited = (
    combined_target
    .merge(
        audit_dataframe,
        on="path",
        how="left",
        validate="one_to_one"
    )
)


if (
    combined_audited[
        "audit_status"
    ]
    .isna()
    .any()
):

    raise RuntimeError(
        "Terdapat citra tanpa audit_status."
    )


dataset_a_manifest = (
    combined_audited[
        combined_audited[
            "dataset"
        ] == "Dataset_A"
    ]
    .copy()
    .reset_index(drop=True)
)


dataset_b_manifest = (
    combined_audited[
        combined_audited[
            "dataset"
        ] == "Dataset_B"
    ]
    .copy()
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 7. SUMMARY
# ------------------------------------------------------------

audit_status_distribution = (
    combined_audited[
        "audit_status"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "audit_status"
    )
    .reset_index(
        name="jumlah"
    )
)


print(
    "\nAUDIT CITRA SELESAI / COMPLETE"
)

display(
    audit_status_distribution
)


print("\nJumlah manifest:")

print(
    "Dataset A:",
    len(dataset_a_manifest)
)

print(
    "Dataset B:",
    len(dataset_b_manifest)
)

print(
    "Gabungan :",
    len(combined_audited)
)

print(
    "\nCheckpoint:",
    AUDIT_RAW_PATH
)

Jumlah citra yang harus diaudit: 3545
Checkpoint ditemukan: 3545 citra sudah selesai
Citra yang belum diaudit: 0


Mengaudit citra: 0citra [00:00, ?citra/s]


AUDIT CITRA SELESAI / COMPLETE


,audit_status,jumlah
0,valid,3545



Jumlah manifest:
Dataset A: 3305
Dataset B: 240
Gabungan : 3545

Checkpoint: /content/drive/MyDrive/eksperimen_baru/baseline_v2/logs/image_audit_raw.csv


In [ ]:
# ============================================================
# CELL 15A — FINAL AUDIT CHECKPOINT GATE
# ============================================================

AUDITED_CHECKPOINT_PATH = (
    LOGS_DIR
    / "combined_audited_checkpoint.csv"
)


required_audit_columns = {
    "path",
    "file_size_bytes",
    "width",
    "height",
    "aspect_ratio",
    "color_mode",
    "image_format",
    "sha256",
    "phash",
    "audit_status",
    "audit_error"
}


missing_audit_columns = (
    required_audit_columns
    -
    set(
        audit_dataframe.columns
    )
)


if missing_audit_columns:

    raise RuntimeError(
        "Audit dataframe belum lengkap.\n"
        "Kolom yang hilang: "
        f"{sorted(missing_audit_columns)}"
    )


if (
    len(audit_dataframe)
    !=
    len(combined_target)
):

    raise RuntimeError(
        "Audit belum complete. "
        "Final checkpoint tidak boleh dibuat."
    )


if (
    audit_dataframe[
        "path"
    ]
    .duplicated()
    .any()
):

    raise RuntimeError(
        "Terdapat duplicate path "
        "pada audit_dataframe."
    )


if (
    combined_audited[
        "path"
    ]
    .duplicated()
    .any()
):

    raise RuntimeError(
        "Terdapat duplicate path "
        "pada combined_audited."
    )


if (
    combined_audited[
        "audit_status"
    ]
    .isna()
    .any()
):

    raise RuntimeError(
        "Terdapat audit_status kosong."
    )


# ------------------------------------------------------------
# Save final checkpoints
# ------------------------------------------------------------

audit_dataframe.to_csv(
    AUDIT_RAW_PATH,
    index=False
)


combined_audited.to_csv(
    AUDITED_CHECKPOINT_PATH,
    index=False
)


# ------------------------------------------------------------
# Reload verification
# ------------------------------------------------------------

audit_reload = pd.read_csv(
    AUDIT_RAW_PATH
)

combined_reload = pd.read_csv(
    AUDITED_CHECKPOINT_PATH
)


if (
    len(audit_reload)
    !=
    len(audit_dataframe)
):

    raise RuntimeError(
        "Verifikasi image_audit_raw.csv gagal."
    )


if (
    len(combined_reload)
    !=
    len(combined_audited)
):

    raise RuntimeError(
        "Verifikasi combined checkpoint gagal."
    )


print(
    "Final audit checkpoint berhasil "
    "disimpan dan diverifikasi:"
)

print(
    "-",
    AUDIT_RAW_PATH
)

print(
    "-",
    AUDITED_CHECKPOINT_PATH
)

print(
    "\nJumlah rows:",
    len(audit_dataframe)
)

Final audit checkpoint berhasil disimpan dan diverifikasi:
- /content/drive/MyDrive/eksperimen_baru/baseline_v2/logs/image_audit_raw.csv
- /content/drive/MyDrive/eksperimen_baru/baseline_v2/logs/combined_audited_checkpoint.csv

Jumlah rows: 3545


In [ ]:
# ============================================================
# CELL 16 — MENYIMPAN MANIFEST HASIL AUDIT
# ============================================================

DATASET_A_ALL_PATH = (
    MANIFEST_DIR
    / "dataset_a_all_images.csv"
)

DATASET_B_ALL_PATH = (
    MANIFEST_DIR
    / "dataset_b_all_images.csv"
)

DATASET_A_MANIFEST_PATH = (
    MANIFEST_DIR
    / "dataset_a_manifest.csv"
)

DATASET_B_MANIFEST_PATH = (
    MANIFEST_DIR
    / "dataset_b_manifest.csv"
)

COMBINED_MANIFEST_PATH = (
    MANIFEST_DIR
    / "combined_target_manifest.csv"
)

dataset_a_all.to_csv(
    DATASET_A_ALL_PATH,
    index=False
)

dataset_b_all.to_csv(
    DATASET_B_ALL_PATH,
    index=False
)

dataset_a_manifest.to_csv(
    DATASET_A_MANIFEST_PATH,
    index=False
)

dataset_b_manifest.to_csv(
    DATASET_B_MANIFEST_PATH,
    index=False
)

combined_audited.to_csv(
    COMBINED_MANIFEST_PATH,
    index=False
)

print("Manifest berhasil disimpan:")
print("-", DATASET_A_ALL_PATH)
print("-", DATASET_B_ALL_PATH)
print("-", DATASET_A_MANIFEST_PATH)
print("-", DATASET_B_MANIFEST_PATH)
print("-", COMBINED_MANIFEST_PATH)

Manifest berhasil disimpan:
- /content/drive/MyDrive/eksperimen_baru/baseline_v2/manifests/dataset_a_all_images.csv
- /content/drive/MyDrive/eksperimen_baru/baseline_v2/manifests/dataset_b_all_images.csv
- /content/drive/MyDrive/eksperimen_baru/baseline_v2/manifests/dataset_a_manifest.csv
- /content/drive/MyDrive/eksperimen_baru/baseline_v2/manifests/dataset_b_manifest.csv
- /content/drive/MyDrive/eksperimen_baru/baseline_v2/manifests/combined_target_manifest.csv


In [ ]:
# ============================================================
# CELL 17 — PEMERIKSAAN CITRA RUSAK
# ============================================================

corrupt_images = (
    combined_audited[
        combined_audited[
            "audit_status"
        ] != "valid"
    ]
    .copy()
    .reset_index(drop=True)
)

CORRUPT_IMAGES_PATH = (
    METRICS_DIR
    / "corrupt_images.csv"
)

corrupt_images.to_csv(
    CORRUPT_IMAGES_PATH,
    index=False
)

print(
    "Jumlah citra rusak/error:",
    len(corrupt_images)
)

if len(corrupt_images) > 0:
    display(
        corrupt_images[
            [
                "dataset",
                "relative_path",
                "label",
                "audit_status",
                "audit_error"
            ]
        ]
    )
else:
    print(
        "Tidak ditemukan citra rusak."
    )

Jumlah citra rusak/error: 0
Tidak ditemukan citra rusak.


In [ ]:
# ============================================================
# CELL 18 — EXACT DUPLICATE BERDASARKAN SHA-256
# ============================================================

valid_images = (
    combined_audited[
        combined_audited[
            "audit_status"
        ] == "valid"
    ]
    .copy()
    .reset_index(drop=True)
)

exact_duplicate_mask = (
    valid_images[
        "sha256"
    ].duplicated(
        keep=False
    )
)

exact_duplicates = (
    valid_images[
        exact_duplicate_mask
    ]
    .copy()
    .sort_values(
        [
            "sha256",
            "dataset",
            "label",
            "relative_path"
        ]
    )
    .reset_index(drop=True)
)

if not exact_duplicates.empty:
    exact_duplicates[
        "exact_duplicate_group"
    ] = (
        exact_duplicates
        .groupby("sha256")
        .ngroup()
        + 1
    )

    dataset_count_per_hash = (
        exact_duplicates
        .groupby("sha256")[
            "dataset"
        ]
        .nunique()
        .to_dict()
    )

    exact_duplicates[
        "duplicate_scope"
    ] = exact_duplicates[
        "sha256"
    ].map(
        lambda hash_value: (
            "cross_dataset"
            if dataset_count_per_hash[
                hash_value
            ] > 1
            else "within_dataset"
        )
    )

EXACT_DUPLICATES_PATH = (
    METRICS_DIR
    / "exact_duplicate_images.csv"
)

exact_duplicates.to_csv(
    EXACT_DUPLICATES_PATH,
    index=False
)

print(
    "Jumlah baris exact duplicate:",
    len(exact_duplicates)
)

if len(exact_duplicates) > 0:
    display(
        exact_duplicates[
            [
                "exact_duplicate_group",
                "duplicate_scope",
                "dataset",
                "label",
                "relative_path",
                "sha256"
            ]
        ].head(50)
    )
else:
    print(
        "Tidak ditemukan exact duplicate."
    )

Jumlah baris exact duplicate: 52


,exact_duplicate_group,duplicate_scope,dataset,label,relative_path,sha256
0,1,within_dataset,Dataset_A,tungro,tungro/104116.jpg,0793d3c8d1b3f6933ee9f9d4aebe5ae03a35fd3fc8a981...
1,1,within_dataset,Dataset_A,tungro,tungro/106818.jpg,0793d3c8d1b3f6933ee9f9d4aebe5ae03a35fd3fc8a981...
2,2,within_dataset,Dataset_A,blast,blast/100501.jpg,16975d80ca86d4b305d6b25f2d0946173a9938dc7af4c1...
3,2,within_dataset,Dataset_A,blast,blast/107069.jpg,16975d80ca86d4b305d6b25f2d0946173a9938dc7af4c1...
4,3,within_dataset,Dataset_A,blast,blast/106102.jpg,1d5791c36c0fb7862d4a5f92b7786b8bc481e040def6a6...
5,3,within_dataset,Dataset_A,blast,blast/106520.jpg,1d5791c36c0fb7862d4a5f92b7786b8bc481e040def6a6...
6,4,within_dataset,Dataset_A,blast,blast/104586.jpg,2d5c890dd1c119104478339cbe73dee3201acc3e35510a...
7,4,within_dataset,Dataset_A,blast,blast/105626.jpg,2d5c890dd1c119104478339cbe73dee3201acc3e35510a...
8,5,within_dataset,Dataset_A,bacterial_leaf_blight,bacterial_leaf_blight/102038.jpg,3395680a96103598798655f0307af062c401d276bad884...
9,5,within_dataset,Dataset_A,bacterial_leaf_blight,bacterial_leaf_blight/108125.jpg,3395680a96103598798655f0307af062c401d276bad884...


In [ ]:
# ============================================================
# CELL 19 — RINGKASAN KELOMPOK EXACT DUPLICATE
# ============================================================

if exact_duplicates.empty:
    exact_group_summary = pd.DataFrame(
        columns=[
            "sha256",
            "group_size",
            "dataset_count",
            "label_count",
            "datasets",
            "labels",
            "duplicate_scope",
            "excess_images"
        ]
    )

    print("Tidak ditemukan exact duplicate.")

else:
    exact_group_summary = (
        exact_duplicates
        .groupby("sha256")
        .agg(
            group_size=("path", "size"),
            dataset_count=("dataset", "nunique"),
            label_count=("label", "nunique"),
            datasets=(
                "dataset",
                lambda values: ", ".join(
                    sorted(set(values))
                )
            ),
            labels=(
                "label",
                lambda values: ", ".join(
                    sorted(set(values))
                )
            )
        )
        .reset_index()
    )

    exact_group_summary[
        "duplicate_scope"
    ] = exact_group_summary[
        "dataset_count"
    ].map(
        lambda value: (
            "cross_dataset"
            if value > 1
            else "within_dataset"
        )
    )

    exact_group_summary[
        "excess_images"
    ] = (
        exact_group_summary[
            "group_size"
        ] - 1
    )

    exact_group_summary = (
        exact_group_summary
        .sort_values(
            [
                "group_size",
                "sha256"
            ],
            ascending=[
                False,
                True
            ]
        )
        .reset_index(drop=True)
    )

display(exact_group_summary)

print("\nRingkasan:")
print(
    "Jumlah kelompok exact duplicate :",
    len(exact_group_summary)
)

print(
    "Jumlah baris dalam kelompok      :",
    int(
        exact_group_summary[
            "group_size"
        ].sum()
    )
)

print(
    "Jumlah citra berlebih            :",
    int(
        exact_group_summary[
            "excess_images"
        ].sum()
    )
)

print(
    "Kelompok lintas dataset          :",
    int(
        (
            exact_group_summary[
                "duplicate_scope"
            ] == "cross_dataset"
        ).sum()
    )
)

GROUP_SUMMARY_PATH = (
    METRICS_DIR
    / "exact_duplicate_group_summary.csv"
)

exact_group_summary.to_csv(
    GROUP_SUMMARY_PATH,
    index=False
)

print("\nDisimpan di:")
print(GROUP_SUMMARY_PATH)

,sha256,group_size,dataset_count,label_count,datasets,labels,duplicate_scope,excess_images
0,0793d3c8d1b3f6933ee9f9d4aebe5ae03a35fd3fc8a981...,2,1,1,Dataset_A,tungro,within_dataset,1
1,16975d80ca86d4b305d6b25f2d0946173a9938dc7af4c1...,2,1,1,Dataset_A,blast,within_dataset,1
2,1d5791c36c0fb7862d4a5f92b7786b8bc481e040def6a6...,2,1,1,Dataset_A,blast,within_dataset,1
3,2d5c890dd1c119104478339cbe73dee3201acc3e35510a...,2,1,1,Dataset_A,blast,within_dataset,1
4,3395680a96103598798655f0307af062c401d276bad884...,2,1,1,Dataset_A,bacterial_leaf_blight,within_dataset,1
5,3c5140b9762a0ae47c2cbfc03010dc4081eb74ac8b23d4...,2,1,1,Dataset_A,tungro,within_dataset,1
6,438409c82958d43b4cb2624241951ac5157e3d10d74ad1...,2,1,1,Dataset_A,bacterial_leaf_blight,within_dataset,1
7,64861fb5f76272e8dd8b561c42f8feac8917b941a6639d...,2,1,1,Dataset_A,tungro,within_dataset,1
8,6815f60ff06a892e37f47eb419c531532389d42bb4589b...,2,1,1,Dataset_A,bacterial_leaf_blight,within_dataset,1
9,684af4a1c98ac9af8306c73577d0bf4536ac5e389c1761...,2,1,1,Dataset_A,blast,within_dataset,1



Ringkasan:
Jumlah kelompok exact duplicate : 26
Jumlah baris dalam kelompok      : 52
Jumlah citra berlebih            : 26
Kelompok lintas dataset          : 0

Disimpan di:
/content/drive/MyDrive/eksperimen_baru/baseline_v2/metrics/exact_duplicate_group_summary.csv


In [ ]:
# ============================================================
# CELL 20 — KONFLIK LABEL EXACT DUPLICATE
# ============================================================

if exact_duplicates.empty:
    exact_cross_label_conflicts = (
        exact_duplicates.copy()
    )

else:
    label_count_per_hash = (
        exact_duplicates
        .groupby("sha256")[
            "label"
        ]
        .nunique()
    )

    conflicting_hashes = (
        label_count_per_hash[
            label_count_per_hash > 1
        ]
        .index
    )

    exact_cross_label_conflicts = (
        exact_duplicates[
            exact_duplicates[
                "sha256"
            ].isin(
                conflicting_hashes
            )
        ]
        .copy()
        .sort_values(
            [
                "sha256",
                "dataset",
                "label",
                "relative_path"
            ]
        )
        .reset_index(drop=True)
    )

EXACT_CONFLICT_PATH = (
    METRICS_DIR
    / "exact_duplicate_label_conflicts.csv"
)

exact_cross_label_conflicts.to_csv(
    EXACT_CONFLICT_PATH,
    index=False
)

print(
    "Jumlah baris duplicate dengan label berbeda:",
    len(exact_cross_label_conflicts)
)

if len(exact_cross_label_conflicts) > 0:
    print(
        "\nAUDIT DIBLOKIR: ditemukan file identik "
        "dengan label yang berbeda."
    )

    display(
        exact_cross_label_conflicts[
            [
                "dataset",
                "label",
                "relative_path",
                "sha256"
            ]
        ]
    )

else:
    print(
        "Tidak ditemukan konflik label "
        "pada exact duplicate."
    )

Jumlah baris duplicate dengan label berbeda: 0
Tidak ditemukan konflik label pada exact duplicate.


In [ ]:
# ============================================================
# CELL 21 — RENCANA EXACT DEDUPLICATION
# ============================================================

if len(exact_cross_label_conflicts) > 0:
    raise ValueError(
        "Deduplikasi dihentikan karena terdapat "
        "exact duplicate dengan konflik label."
    )

exact_dedup_plan = (
    valid_images
    .copy()
    .sort_values(
        [
            "sha256",
            "dataset",
            "label",
            "relative_path"
        ]
    )
    .reset_index(drop=True)
)

exact_dedup_plan[
    "duplicate_group_size"
] = (
    exact_dedup_plan
    .groupby("sha256")[
        "sha256"
    ]
    .transform("size")
)

exact_dedup_plan[
    "duplicate_rank"
] = (
    exact_dedup_plan
    .groupby("sha256")
    .cumcount()
)

exact_dedup_plan[
    "exact_dedup_action"
] = np.where(
    exact_dedup_plan[
        "duplicate_group_size"
    ] == 1,
    "unique",
    np.where(
        exact_dedup_plan[
            "duplicate_rank"
        ] == 0,
        "keep_representative",
        "remove_duplicate"
    )
)

exact_dedup_plan[
    "exact_duplicate_detected"
] = (
    exact_dedup_plan[
        "duplicate_group_size"
    ] > 1
)

EXACT_DEDUP_PLAN_PATH = (
    METRICS_DIR
    / "exact_deduplication_plan.csv"
)

exact_dedup_plan.to_csv(
    EXACT_DEDUP_PLAN_PATH,
    index=False
)

dedup_action_summary = (
    exact_dedup_plan[
        "exact_dedup_action"
    ]
    .value_counts()
    .rename_axis(
        "exact_dedup_action"
    )
    .reset_index(
        name="jumlah"
    )
)

display(dedup_action_summary)

print("\nRencana deduplikasi disimpan di:")
print(EXACT_DEDUP_PLAN_PATH)

,exact_dedup_action,jumlah
0,unique,3493
1,keep_representative,26
2,remove_duplicate,26



Rencana deduplikasi disimpan di:
/content/drive/MyDrive/eksperimen_baru/baseline_v2/metrics/exact_deduplication_plan.csv


In [ ]:
# ============================================================
# CELL 22 — DAFTAR CITRA YANG DIKELUARKAN
# ============================================================

exact_duplicate_removals = (
    exact_dedup_plan[
        exact_dedup_plan[
            "exact_dedup_action"
        ] == "remove_duplicate"
    ]
    .copy()
    .reset_index(drop=True)
)

EXACT_REMOVALS_PATH = (
    METRICS_DIR
    / "exact_duplicate_removals.csv"
)

exact_duplicate_removals.to_csv(
    EXACT_REMOVALS_PATH,
    index=False
)

print(
    "Jumlah citra yang dikeluarkan dari manifest:",
    len(exact_duplicate_removals)
)

if not exact_duplicate_removals.empty:
    display(
        exact_duplicate_removals[
            [
                "dataset",
                "label",
                "relative_path",
                "sha256",
                "duplicate_group_size",
                "duplicate_rank",
                "exact_dedup_action"
            ]
        ].head(60)
    )

print("\nDaftar removal disimpan di:")
print(EXACT_REMOVALS_PATH)

Jumlah citra yang dikeluarkan dari manifest: 26


,dataset,label,relative_path,sha256,duplicate_group_size,duplicate_rank,exact_dedup_action
0,Dataset_A,tungro,tungro/106818.jpg,0793d3c8d1b3f6933ee9f9d4aebe5ae03a35fd3fc8a981...,2,1,remove_duplicate
1,Dataset_A,blast,blast/107069.jpg,16975d80ca86d4b305d6b25f2d0946173a9938dc7af4c1...,2,1,remove_duplicate
2,Dataset_A,blast,blast/106520.jpg,1d5791c36c0fb7862d4a5f92b7786b8bc481e040def6a6...,2,1,remove_duplicate
3,Dataset_A,blast,blast/105626.jpg,2d5c890dd1c119104478339cbe73dee3201acc3e35510a...,2,1,remove_duplicate
4,Dataset_A,bacterial_leaf_blight,bacterial_leaf_blight/108125.jpg,3395680a96103598798655f0307af062c401d276bad884...,2,1,remove_duplicate
5,Dataset_A,tungro,tungro/105747.jpg,3c5140b9762a0ae47c2cbfc03010dc4081eb74ac8b23d4...,2,1,remove_duplicate
6,Dataset_A,bacterial_leaf_blight,bacterial_leaf_blight/106984.jpg,438409c82958d43b4cb2624241951ac5157e3d10d74ad1...,2,1,remove_duplicate
7,Dataset_A,tungro,tungro/104027.jpg,64861fb5f76272e8dd8b561c42f8feac8917b941a6639d...,2,1,remove_duplicate
8,Dataset_A,bacterial_leaf_blight,bacterial_leaf_blight/104800.jpg,6815f60ff06a892e37f47eb419c531532389d42bb4589b...,2,1,remove_duplicate
9,Dataset_A,blast,blast/108455.jpg,684af4a1c98ac9af8306c73577d0bf4536ac5e389c1761...,2,1,remove_duplicate



Daftar removal disimpan di:
/content/drive/MyDrive/eksperimen_baru/baseline_v2/metrics/exact_duplicate_removals.csv


In [ ]:
# ============================================================
# CELL 23 — MANIFEST BERSIH SETELAH EXACT DEDUPLICATION
# ============================================================

combined_exact_deduplicated = (
    exact_dedup_plan[
        exact_dedup_plan[
            "exact_dedup_action"
        ] != "remove_duplicate"
    ]
    .copy()
    .reset_index(drop=True)
)

dataset_a_exact_deduplicated = (
    combined_exact_deduplicated[
        combined_exact_deduplicated[
            "dataset"
        ] == "Dataset_A"
    ]
    .copy()
    .reset_index(drop=True)
)

dataset_b_exact_deduplicated = (
    combined_exact_deduplicated[
        combined_exact_deduplicated[
            "dataset"
        ] == "Dataset_B"
    ]
    .copy()
    .reset_index(drop=True)
)

print("Jumlah sebelum exact deduplication:")
print("Dataset A :", len(dataset_a_manifest))
print("Dataset B :", len(dataset_b_manifest))
print("Gabungan  :", len(combined_audited))

print("\nJumlah setelah exact deduplication:")
print(
    "Dataset A :",
    len(dataset_a_exact_deduplicated)
)
print(
    "Dataset B :",
    len(dataset_b_exact_deduplicated)
)
print(
    "Gabungan  :",
    len(combined_exact_deduplicated)
)

print("\nJumlah dikeluarkan:")
print(
    len(combined_audited)
    - len(combined_exact_deduplicated)
)

Jumlah sebelum exact deduplication:
Dataset A : 3305
Dataset B : 240
Gabungan  : 3545

Jumlah setelah exact deduplication:
Dataset A : 3279
Dataset B : 240
Gabungan  : 3519

Jumlah dikeluarkan:
26


In [ ]:
# ============================================================
# CELL 24 — VALIDASI MANIFEST HASIL DEDUPLIKASI
# ============================================================

remaining_exact_duplicate_rows = int(
    combined_exact_deduplicated[
        "sha256"
    ].duplicated(
        keep=False
    ).sum()
)

remaining_duplicate_paths = int(
    combined_exact_deduplicated[
        "path"
    ].duplicated().sum()
)

print(
    "Exact duplicate tersisa:",
    remaining_exact_duplicate_rows
)

print(
    "Path duplikat tersisa:",
    remaining_duplicate_paths
)

if remaining_exact_duplicate_rows > 0:
    raise ValueError(
        "Masih terdapat exact duplicate "
        "setelah proses deduplikasi."
    )

if remaining_duplicate_paths > 0:
    raise ValueError(
        "Masih terdapat path duplikat."
    )

print(
    "Validasi exact deduplication: LULUS"
)

Exact duplicate tersisa: 0
Path duplikat tersisa: 0
Validasi exact deduplication: LULUS


In [ ]:
# ============================================================
# CELL 25 — MENYIMPAN MANIFEST EXACT-DEDUPLICATED
# ============================================================

DATASET_A_DEDUP_PATH = (
    MANIFEST_DIR
    / "dataset_a_exact_deduplicated.csv"
)

DATASET_B_DEDUP_PATH = (
    MANIFEST_DIR
    / "dataset_b_exact_deduplicated.csv"
)

COMBINED_DEDUP_PATH = (
    MANIFEST_DIR
    / "combined_exact_deduplicated.csv"
)

dataset_a_exact_deduplicated.to_csv(
    DATASET_A_DEDUP_PATH,
    index=False
)

dataset_b_exact_deduplicated.to_csv(
    DATASET_B_DEDUP_PATH,
    index=False
)

combined_exact_deduplicated.to_csv(
    COMBINED_DEDUP_PATH,
    index=False
)

print(
    "Manifest hasil exact deduplication berhasil disimpan:"
)

print("-", DATASET_A_DEDUP_PATH)
print("-", DATASET_B_DEDUP_PATH)
print("-", COMBINED_DEDUP_PATH)

Manifest hasil exact deduplication berhasil disimpan:
- /content/drive/MyDrive/eksperimen_baru/baseline_v2/manifests/dataset_a_exact_deduplicated.csv
- /content/drive/MyDrive/eksperimen_baru/baseline_v2/manifests/dataset_b_exact_deduplicated.csv
- /content/drive/MyDrive/eksperimen_baru/baseline_v2/manifests/combined_exact_deduplicated.csv


In [ ]:
# ============================================================
# CELL 26 — DISTRIBUSI KELAS SETELAH DEDUPLIKASI
# ============================================================

class_distribution_after_exact_dedup = (
    combined_exact_deduplicated
    .groupby(
        [
            "dataset",
            "label"
        ]
    )
    .size()
    .reset_index(
        name="jumlah_setelah_dedup"
    )
)

class_distribution_before_exact_dedup = (
    combined_audited
    .groupby(
        [
            "dataset",
            "label"
        ]
    )
    .size()
    .reset_index(
        name="jumlah_sebelum_dedup"
    )
)

class_distribution_dedup_comparison = (
    class_distribution_before_exact_dedup
    .merge(
        class_distribution_after_exact_dedup,
        on=[
            "dataset",
            "label"
        ],
        how="outer"
    )
    .fillna(0)
)

class_distribution_dedup_comparison[
    "jumlah_dikeluarkan"
] = (
    class_distribution_dedup_comparison[
        "jumlah_sebelum_dedup"
    ]
    - class_distribution_dedup_comparison[
        "jumlah_setelah_dedup"
    ]
)

CLASS_DEDUP_COMPARISON_PATH = (
    METRICS_DIR
    / "class_distribution_exact_dedup_comparison.csv"
)

class_distribution_dedup_comparison.to_csv(
    CLASS_DEDUP_COMPARISON_PATH,
    index=False
)

display(
    class_distribution_dedup_comparison
)

print("\nDisimpan di:")
print(CLASS_DEDUP_COMPARISON_PATH)

,dataset,label,jumlah_sebelum_dedup,jumlah_setelah_dedup,jumlah_dikeluarkan
0,Dataset_A,bacterial_leaf_blight,479,471,8
1,Dataset_A,blast,1738,1728,10
2,Dataset_A,tungro,1088,1080,8
3,Dataset_B,bacterial_leaf_blight,80,80,0
4,Dataset_B,blast,80,80,0
5,Dataset_B,tungro,80,80,0



Disimpan di:
/content/drive/MyDrive/eksperimen_baru/baseline_v2/metrics/class_distribution_exact_dedup_comparison.csv


In [ ]:
# ============================================================
# CELL 27 — NEAR-DUPLICATE pHASH SCREENING
# ============================================================

valid_images_clean = (
    combined_exact_deduplicated
    .copy()
    .reset_index(drop=True)
)

PHASH_HAMMING_THRESHOLD = int(
    CONFIG["data_audit"]["phash_hamming_threshold"]
)

if not 0 <= PHASH_HAMMING_THRESHOLD <= 64:
    raise ValueError(
        "pHash Hamming threshold harus berada pada rentang 0–64."
    )

print(
    "pHash Hamming threshold:",
    PHASH_HAMMING_THRESHOLD
)


# ------------------------------------------------------------
# 1. VALIDASI pHASH
# ------------------------------------------------------------

if valid_images_clean["phash"].isna().any():
    raise RuntimeError(
        "Terdapat nilai pHash kosong pada manifest valid."
    )

phash_strings = (
    valid_images_clean["phash"]
    .astype(str)
    .str.strip()
    .tolist()
)

if any(value == "" for value in phash_strings):
    raise RuntimeError(
        "Terdapat nilai pHash kosong."
    )

try:
    phash_integers = [
        int(value, 16)
        for value in phash_strings
    ]
except ValueError as error:
    raise RuntimeError(
        "Format hexadecimal pHash tidak valid."
    ) from error


number_of_images = len(valid_images_clean)


# ------------------------------------------------------------
# 2. UNION-FIND
#
# review_parent:
# seluruh kandidat near-duplicate untuk visual review
#
# split_parent:
# hanya kandidat same-dataset + same-label
# yang boleh masuk satu split group
# ------------------------------------------------------------

review_parent = list(range(number_of_images))
split_parent = list(range(number_of_images))


def find_root(parent, index):
    while parent[index] != index:
        parent[index] = parent[parent[index]]
        index = parent[index]
    return index


def union_nodes(parent, first_index, second_index):
    root_a = find_root(parent, first_index)
    root_b = find_root(parent, second_index)

    if root_a != root_b:
        parent[root_b] = root_a


datasets = (
    valid_images_clean["dataset"]
    .astype(str)
    .to_numpy()
)

labels = (
    valid_images_clean["label"]
    .astype(str)
    .to_numpy()
)

paths = (
    valid_images_clean["path"]
    .astype(str)
    .to_numpy()
)

relative_paths = (
    valid_images_clean["relative_path"]
    .astype(str)
    .to_numpy()
)


# ------------------------------------------------------------
# 3. pHASH HAMMING SCREENING
# ------------------------------------------------------------

candidate_pair_records = []

candidate_indices = set()
cross_dataset_indices = set()
cross_label_indices = set()


print(
    "Mencari kandidat near-duplicate..."
)


for first_index in tqdm(
    range(number_of_images - 1),
    desc="pHash screening",
    unit="image"
):

    first_hash = phash_integers[first_index]

    for second_index in range(
        first_index + 1,
        number_of_images
    ):

        hamming_distance = (
            first_hash
            ^
            phash_integers[second_index]
        ).bit_count()

        if (
            hamming_distance
            >
            PHASH_HAMMING_THRESHOLD
        ):
            continue


        same_dataset = (
            datasets[first_index]
            ==
            datasets[second_index]
        )

        same_label = (
            labels[first_index]
            ==
            labels[second_index]
        )


        candidate_indices.update(
            [
                first_index,
                second_index
            ]
        )


        # Semua kandidat masuk review group
        union_nodes(
            review_parent,
            first_index,
            second_index
        )


        # Hanya same-dataset + same-label
        # yang masuk safe split group
        if same_dataset and same_label:
            union_nodes(
                split_parent,
                first_index,
                second_index
            )


        if not same_dataset:
            cross_dataset_indices.update(
                [
                    first_index,
                    second_index
                ]
            )


        if not same_label:
            cross_label_indices.update(
                [
                    first_index,
                    second_index
                ]
            )


        candidate_pair_records.append(
            {
                "index_a": int(first_index),
                "index_b": int(second_index),

                "path_a": paths[first_index],
                "path_b": paths[second_index],

                "relative_path_a":
                    relative_paths[first_index],

                "relative_path_b":
                    relative_paths[second_index],

                "dataset_a":
                    datasets[first_index],

                "dataset_b":
                    datasets[second_index],

                "label_a":
                    labels[first_index],

                "label_b":
                    labels[second_index],

                "phash_a":
                    phash_strings[first_index],

                "phash_b":
                    phash_strings[second_index],

                "hamming_distance":
                    int(hamming_distance),

                "cross_dataset":
                    bool(not same_dataset),

                "cross_label":
                    bool(not same_label)
            }
        )


# ------------------------------------------------------------
# 4. COMPONENT MAPPING
# ------------------------------------------------------------

def create_component_mapping(
    parent,
    indices,
    prefix
):

    root_members = {}

    for index in indices:

        root = find_root(
            parent,
            index
        )

        root_members.setdefault(
            root,
            []
        ).append(index)


    root_members = {
        root: members
        for root, members
        in root_members.items()
        if len(members) > 1
    }


    index_to_group = {}


    sorted_components = sorted(
        root_members.items(),
        key=lambda item: min(item[1])
    )


    for group_number, (
        _,
        members
    ) in enumerate(
        sorted_components,
        start=1
    ):

        group_id = (
            f"{prefix}_"
            f"{group_number:04d}"
        )

        for index in members:
            index_to_group[index] = group_id


    return (
        index_to_group,
        root_members
    )


review_group_mapping, review_components = (
    create_component_mapping(
        review_parent,
        candidate_indices,
        "review"
    )
)


split_group_mapping, split_components = (
    create_component_mapping(
        split_parent,
        candidate_indices,
        "near_dup"
    )
)


# ------------------------------------------------------------
# 5. PAIR-LEVEL ARTIFACT
# ------------------------------------------------------------

PAIR_COLUMNS = [
    "index_a",
    "index_b",

    "path_a",
    "path_b",

    "relative_path_a",
    "relative_path_b",

    "dataset_a",
    "dataset_b",

    "label_a",
    "label_b",

    "phash_a",
    "phash_b",

    "hamming_distance",

    "cross_dataset",
    "cross_label"
]


perceptual_candidate_pairs = pd.DataFrame(
    candidate_pair_records,
    columns=PAIR_COLUMNS
)


if not perceptual_candidate_pairs.empty:

    perceptual_candidate_pairs[
        "review_group_id"
    ] = (
        perceptual_candidate_pairs[
            "index_a"
        ]
        .map(
            review_group_mapping
        )
    )

else:

    perceptual_candidate_pairs[
        "review_group_id"
    ] = pd.Series(
        dtype="object"
    )


PERCEPTUAL_PAIR_PATH = (
    METRICS_DIR
    / "perceptual_duplicate_pairs.csv"
)


perceptual_candidate_pairs.to_csv(
    PERCEPTUAL_PAIR_PATH,
    index=False
)


# ------------------------------------------------------------
# 6. CROSS-LABEL ARTIFACT
# Tetap dibuat walaupun 0 rows
# ------------------------------------------------------------

phash_label_conflicts = (
    perceptual_candidate_pairs[
        perceptual_candidate_pairs[
            "cross_label"
        ] == True
    ]
    .copy()
    .reset_index(drop=True)
)


PHASH_CONFLICT_PATH = (
    METRICS_DIR
    / "phash_label_conflicts.csv"
)


phash_label_conflicts.to_csv(
    PHASH_CONFLICT_PATH,
    index=False
)


# ------------------------------------------------------------
# 7. CROSS-DATASET ARTIFACT
# ------------------------------------------------------------

phash_cross_dataset_candidates = (
    perceptual_candidate_pairs[
        perceptual_candidate_pairs[
            "cross_dataset"
        ] == True
    ]
    .copy()
    .reset_index(drop=True)
)


PHASH_CROSS_DATASET_PATH = (
    METRICS_DIR
    / "phash_cross_dataset_candidates.csv"
)


phash_cross_dataset_candidates.to_csv(
    PHASH_CROSS_DATASET_PATH,
    index=False
)


# ------------------------------------------------------------
# 8. IMAGE-LEVEL CANDIDATE TABLE
# ------------------------------------------------------------

candidate_indices_sorted = sorted(
    candidate_indices
)


if candidate_indices_sorted:

    perceptual_candidates = (
        valid_images_clean
        .iloc[
            candidate_indices_sorted
        ]
        .copy()
    )


    # index asli dari valid_images_clean
    perceptual_candidates[
        "source_index"
    ] = (
        perceptual_candidates.index
    )


    perceptual_candidates[
        "review_group_id"
    ] = (
        perceptual_candidates[
            "source_index"
        ]
        .map(
            review_group_mapping
        )
    )


    perceptual_candidates[
        "near_duplicate_group_id"
    ] = (
        perceptual_candidates[
            "source_index"
        ]
        .map(
            split_group_mapping
        )
    )


    perceptual_candidates[
        "candidate_scope"
    ] = (
        perceptual_candidates[
            "source_index"
        ]
        .map(
            lambda index:
            (
                "cross_dataset"
                if index
                in cross_dataset_indices
                else
                "within_dataset"
            )
        )
    )


    perceptual_candidates[
        "candidate_label_conflict"
    ] = (
        perceptual_candidates[
            "source_index"
        ]
        .isin(
            cross_label_indices
        )
    )


    pair_count_by_index = {
        index: 0
        for index
        in candidate_indices_sorted
    }


    minimum_distance_by_index = {
        index: None
        for index
        in candidate_indices_sorted
    }


    for record in candidate_pair_records:

        distance = record[
            "hamming_distance"
        ]

        for index in [
            record["index_a"],
            record["index_b"]
        ]:

            pair_count_by_index[
                index
            ] += 1


            current_minimum = (
                minimum_distance_by_index[
                    index
                ]
            )


            if (
                current_minimum is None
                or
                distance < current_minimum
            ):

                minimum_distance_by_index[
                    index
                ] = distance


    perceptual_candidates[
        "candidate_pair_count"
    ] = (
        perceptual_candidates[
            "source_index"
        ]
        .map(
            pair_count_by_index
        )
    )


    perceptual_candidates[
        "min_hamming_distance"
    ] = (
        perceptual_candidates[
            "source_index"
        ]
        .map(
            minimum_distance_by_index
        )
    )


    perceptual_candidates = (
        perceptual_candidates
        .reset_index(drop=True)
    )


else:

    perceptual_candidates = (
        valid_images_clean
        .iloc[0:0]
        .copy()
    )

    perceptual_candidates[
        "source_index"
    ] = pd.Series(
        dtype="int64"
    )

    perceptual_candidates[
        "review_group_id"
    ] = pd.Series(
        dtype="object"
    )

    perceptual_candidates[
        "near_duplicate_group_id"
    ] = pd.Series(
        dtype="object"
    )

    perceptual_candidates[
        "candidate_scope"
    ] = pd.Series(
        dtype="object"
    )

    perceptual_candidates[
        "candidate_label_conflict"
    ] = pd.Series(
        dtype="bool"
    )

    perceptual_candidates[
        "candidate_pair_count"
    ] = pd.Series(
        dtype="int64"
    )

    perceptual_candidates[
        "min_hamming_distance"
    ] = pd.Series(
        dtype="float64"
    )


PERCEPTUAL_CANDIDATES_PATH = (
    METRICS_DIR
    / "perceptual_duplicate_candidates.csv"
)


perceptual_candidates.to_csv(
    PERCEPTUAL_CANDIDATES_PATH,
    index=False
)


# ------------------------------------------------------------
# 9. SUMMARY
# ------------------------------------------------------------

print(
    "\nNEAR-DUPLICATE pHASH SUMMARY"
)

print(
    "Candidate pairs       :",
    len(perceptual_candidate_pairs)
)

print(
    "Candidate images      :",
    len(perceptual_candidates)
)

print(
    "Visual review groups  :",
    len(
        set(
            review_group_mapping.values()
        )
    )
)

print(
    "Safe split groups     :",
    len(
        set(
            split_group_mapping.values()
        )
    )
)

print(
    "Cross-dataset pairs   :",
    len(
        phash_cross_dataset_candidates
    )
)

print(
    "Cross-label pairs     :",
    len(
        phash_label_conflicts
    )
)


print("\nArtifacts:")

print(
    "-",
    PERCEPTUAL_PAIR_PATH
)

print(
    "-",
    PERCEPTUAL_CANDIDATES_PATH
)

print(
    "-",
    PHASH_CONFLICT_PATH
)

print(
    "-",
    PHASH_CROSS_DATASET_PATH
)

pHash Hamming threshold: 4
Mencari kandidat near-duplicate...


pHash screening:   0%|          | 0/3518 [00:00<?, ?image/s]


NEAR-DUPLICATE pHASH SUMMARY
Candidate pairs       : 741
Candidate images      : 814
Visual review groups  : 285
Safe split groups     : 284
Cross-dataset pairs   : 0
Cross-label pairs     : 2

Artifacts:
- /content/drive/MyDrive/eksperimen_baru/baseline_v2/metrics/perceptual_duplicate_pairs.csv
- /content/drive/MyDrive/eksperimen_baru/baseline_v2/metrics/perceptual_duplicate_candidates.csv
- /content/drive/MyDrive/eksperimen_baru/baseline_v2/metrics/phash_label_conflicts.csv
- /content/drive/MyDrive/eksperimen_baru/baseline_v2/metrics/phash_cross_dataset_candidates.csv


In [ ]:
# ============================================================
# CELL 28 — RINGKASAN KELOMPOK NEAR-DUPLICATE
# ============================================================

if perceptual_candidates.empty:

    perceptual_group_summary = pd.DataFrame(
        columns=[
            "review_group_id",
            "group_size",
            "dataset_count",
            "label_count",
            "datasets",
            "labels",
            "candidate_scope",
            "candidate_excess_rows",
            "min_hamming_distance",
            "max_hamming_distance",
            "candidate_pair_count"
        ]
    )

    print(
        "Tidak ditemukan kandidat near-duplicate."
    )


else:

    perceptual_group_summary = (
        perceptual_candidates
        .groupby(
            "review_group_id",
            dropna=False
        )
        .agg(

            group_size=(
                "path",
                "size"
            ),

            dataset_count=(
                "dataset",
                "nunique"
            ),

            label_count=(
                "label",
                "nunique"
            ),

            datasets=(
                "dataset",
                lambda values:
                ", ".join(
                    sorted(
                        set(values)
                    )
                )
            ),

            labels=(
                "label",
                lambda values:
                ", ".join(
                    sorted(
                        set(values)
                    )
                )
            )
        )
        .reset_index()
    )


    perceptual_group_summary[
        "candidate_scope"
    ] = (
        perceptual_group_summary[
            "dataset_count"
        ]
        .map(
            lambda count:
            (
                "cross_dataset"
                if count > 1
                else
                "within_dataset"
            )
        )
    )


    perceptual_group_summary[
        "candidate_excess_rows"
    ] = (
        perceptual_group_summary[
            "group_size"
        ]
        - 1
    )


    if not perceptual_candidate_pairs.empty:

        pair_statistics = (
            perceptual_candidate_pairs
            .groupby(
                "review_group_id"
            )
            .agg(

                min_hamming_distance=(
                    "hamming_distance",
                    "min"
                ),

                max_hamming_distance=(
                    "hamming_distance",
                    "max"
                ),

                candidate_pair_count=(
                    "hamming_distance",
                    "size"
                )
            )
            .reset_index()
        )


        perceptual_group_summary = (
            perceptual_group_summary
            .merge(
                pair_statistics,
                on="review_group_id",
                how="left",
                validate="one_to_one"
            )
        )


    else:

        perceptual_group_summary[
            "min_hamming_distance"
        ] = np.nan

        perceptual_group_summary[
            "max_hamming_distance"
        ] = np.nan

        perceptual_group_summary[
            "candidate_pair_count"
        ] = 0


    perceptual_group_summary = (
        perceptual_group_summary
        .sort_values(
            [
                "group_size",
                "review_group_id"
            ],
            ascending=[
                False,
                True
            ]
        )
        .reset_index(drop=True)
    )


PERCEPTUAL_GROUP_SUMMARY_PATH = (
    METRICS_DIR
    / "perceptual_candidate_group_summary.csv"
)


perceptual_group_summary.to_csv(
    PERCEPTUAL_GROUP_SUMMARY_PATH,
    index=False
)


display(
    perceptual_group_summary
)


print(
    "\nRingkasan kandidat near-duplicate:"
)


print(
    "Jumlah visual-review groups :",
    len(
        perceptual_group_summary
    )
)


print(
    "Jumlah kandidat citra       :",
    len(
        perceptual_candidates
    )
)


print(
    "Candidate pairs             :",
    len(
        perceptual_candidate_pairs
    )
)


print(
    "Cross-dataset pairs         :",
    len(
        phash_cross_dataset_candidates
    )
)


print(
    "Cross-label pairs           :",
    len(
        phash_label_conflicts
    )
)


print(
    "\nRingkasan disimpan di:"
)

print(
    PERCEPTUAL_GROUP_SUMMARY_PATH
)

,review_group_id,group_size,dataset_count,label_count,datasets,labels,candidate_scope,candidate_excess_rows,min_hamming_distance,max_hamming_distance,candidate_pair_count
0,review_0047,21,1,1,Dataset_A,blast,within_dataset,20,0,4,52
1,review_0050,15,1,1,Dataset_A,blast,within_dataset,14,0,4,32
2,review_0032,9,1,1,Dataset_A,blast,within_dataset,8,0,4,20
3,review_0100,9,1,1,Dataset_A,blast,within_dataset,8,2,4,12
4,review_0108,9,1,1,Dataset_A,blast,within_dataset,8,0,4,15
...,...,...,...,...,...,...,...,...,...,...,...
280,review_0281,2,1,1,Dataset_A,blast,within_dataset,1,4,4,1
281,review_0282,2,1,1,Dataset_A,tungro,within_dataset,1,4,4,1
282,review_0283,2,1,1,Dataset_A,blast,within_dataset,1,4,4,1
283,review_0284,2,1,1,Dataset_A,blast,within_dataset,1,2,2,1



Ringkasan kandidat near-duplicate:
Jumlah visual-review groups : 285
Jumlah kandidat citra       : 814
Candidate pairs             : 741
Cross-dataset pairs         : 0
Cross-label pairs           : 2

Ringkasan disimpan di:
/content/drive/MyDrive/eksperimen_baru/baseline_v2/metrics/perceptual_candidate_group_summary.csv


In [ ]:
# ============================================================
# CELL 29 — VISUAL REVIEW KELOMPOK NEAR-DUPLICATE
# ============================================================

import math
import matplotlib.pyplot as plt
from PIL import Image


PERCEPTUAL_REVIEW_DIR = (
    FIGURES_DIR
    / "perceptual_group_review"
)


PERCEPTUAL_REVIEW_DIR.mkdir(
    parents=True,
    exist_ok=True
)


def save_perceptual_group_review(
    group_dataframe,
    group_id
):

    group_dataframe = (
        group_dataframe
        .sort_values(
            [
                "dataset",
                "label",
                "relative_path"
            ]
        )
        .reset_index(drop=True)
    )


    number_of_images = len(
        group_dataframe
    )


    number_of_columns = min(
        3,
        number_of_images
    )


    number_of_rows = math.ceil(
        number_of_images
        /
        number_of_columns
    )


    figure, axes = plt.subplots(
        number_of_rows,
        number_of_columns,
        figsize=(
            4 * number_of_columns,
            4 * number_of_rows
        ),
        squeeze=False
    )


    axes_flat = axes.flatten()


    for axis in axes_flat:
        axis.axis("off")


    for image_index, row in (
        group_dataframe
        .iterrows()
    ):

        axis = axes_flat[
            image_index
        ]


        with Image.open(
            row["path"]
        ) as image:

            axis.imshow(
                image.convert("RGB")
            )


        axis.set_title(
            (
                f"{row['dataset']} | "
                f"{row['label']}\n"
                f"{row['relative_path']}\n"
                f"pHash: {row['phash']}"
            ),
            fontsize=8
        )


        axis.axis("off")


    figure.suptitle(
        (
            "Near-Duplicate Visual Review\n"
            f"Group: {group_id}"
        ),
        fontsize=13
    )


    figure.tight_layout()


    safe_group_name = (
        str(group_id)
        .replace("/", "_")
        .replace("\\", "_")
    )


    output_path = (
        PERCEPTUAL_REVIEW_DIR
        /
        f"{safe_group_name}.png"
    )


    figure.savefig(
        output_path,
        dpi=180,
        bbox_inches="tight"
    )


    plt.close(
        figure
    )


    return str(
        output_path
    )


# ------------------------------------------------------------
# GENERATE VISUAL REVIEW
# ------------------------------------------------------------

REVIEW_COLUMNS = [
    "review_group_id",
    "group_size",
    "datasets",
    "labels",
    "candidate_scope",
    "candidate_label_conflict",
    "review_image_path",
    "manual_decision",
    "manual_note"
]


review_records = []


if perceptual_candidates.empty:

    print(
        "Tidak ada kelompok near-duplicate "
        "yang perlu divisualisasikan."
    )


else:

    for review_group_id, group_dataframe in (
        perceptual_candidates
        .groupby(
            "review_group_id"
        )
    ):

        saved_path = (
            save_perceptual_group_review(
                group_dataframe=group_dataframe,
                group_id=review_group_id
            )
        )


        review_records.append(
            {
                "review_group_id":
                    review_group_id,

                "group_size":
                    int(
                        len(
                            group_dataframe
                        )
                    ),

                "datasets":
                    ", ".join(
                        sorted(
                            group_dataframe[
                                "dataset"
                            ]
                            .astype(str)
                            .unique()
                        )
                    ),

                "labels":
                    ", ".join(
                        sorted(
                            group_dataframe[
                                "label"
                            ]
                            .astype(str)
                            .unique()
                        )
                    ),

                "candidate_scope":
                    (
                        "cross_dataset"
                        if
                        group_dataframe[
                            "dataset"
                        ].nunique() > 1
                        else
                        "within_dataset"
                    ),

                "candidate_label_conflict":
                    bool(
                        group_dataframe[
                            "label"
                        ].nunique()
                        > 1
                    ),

                "review_image_path":
                    saved_path,

                "manual_decision":
                    "pending_review",

                "manual_note":
                    ""
            }
        )


perceptual_visual_review = (
    pd.DataFrame(
        review_records,
        columns=REVIEW_COLUMNS
    )
)


PERCEPTUAL_VISUAL_REVIEW_PATH = (
    METRICS_DIR
    / "perceptual_visual_review.csv"
)


perceptual_visual_review.to_csv(
    PERCEPTUAL_VISUAL_REVIEW_PATH,
    index=False
)


print(
    "Jumlah visual groups:",
    len(
        perceptual_visual_review
    )
)


print(
    "\nFolder visual review:"
)

print(
    PERCEPTUAL_REVIEW_DIR
)


print(
    "\nTabel review:"
)

print(
    PERCEPTUAL_VISUAL_REVIEW_PATH
)


display(
    perceptual_visual_review.head(10)
)

Jumlah visual groups: 285

Folder visual review:
/content/drive/MyDrive/eksperimen_baru/baseline_v2/figures/perceptual_group_review

Tabel review:
/content/drive/MyDrive/eksperimen_baru/baseline_v2/metrics/perceptual_visual_review.csv


,review_group_id,group_size,datasets,labels,candidate_scope,candidate_label_conflict,review_image_path,manual_decision,manual_note
0,review_0001,6,Dataset_A,blast,within_dataset,False,/content/drive/MyDrive/eksperimen_baru/baselin...,pending_review,
1,review_0002,2,Dataset_A,bacterial_leaf_blight,within_dataset,False,/content/drive/MyDrive/eksperimen_baru/baselin...,pending_review,
2,review_0003,2,Dataset_A,tungro,within_dataset,False,/content/drive/MyDrive/eksperimen_baru/baselin...,pending_review,
3,review_0004,8,Dataset_A,blast,within_dataset,False,/content/drive/MyDrive/eksperimen_baru/baselin...,pending_review,
4,review_0005,8,Dataset_A,blast,within_dataset,False,/content/drive/MyDrive/eksperimen_baru/baselin...,pending_review,
5,review_0006,2,Dataset_A,tungro,within_dataset,False,/content/drive/MyDrive/eksperimen_baru/baselin...,pending_review,
6,review_0007,2,Dataset_A,blast,within_dataset,False,/content/drive/MyDrive/eksperimen_baru/baselin...,pending_review,
7,review_0008,2,Dataset_A,bacterial_leaf_blight,within_dataset,False,/content/drive/MyDrive/eksperimen_baru/baselin...,pending_review,
8,review_0009,4,Dataset_A,tungro,within_dataset,False,/content/drive/MyDrive/eksperimen_baru/baselin...,pending_review,
9,review_0010,3,Dataset_A,tungro,within_dataset,False,/content/drive/MyDrive/eksperimen_baru/baselin...,pending_review,


In [ ]:
# ============================================================
# CELL 29A — RESOLUSI CROSS-LABEL NEAR-DUPLICATE
# CONSERVATIVE EXCLUSION
# ============================================================

CROSS_LABEL_RESOLUTION_PATH = (
    METRICS_DIR
    / "phash_cross_label_resolution.csv"
)


# ------------------------------------------------------------
# 1. IDENTIFIKASI REVIEW GROUP YANG MEMILIKI CROSS-LABEL PAIR
# ------------------------------------------------------------

conflict_review_group_ids = sorted(
    phash_label_conflicts[
        "review_group_id"
    ]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)


print(
    "Cross-label review groups:",
    conflict_review_group_ids
)


if not conflict_review_group_ids:

    print(
        "Tidak ada cross-label near-duplicate "
        "yang perlu diselesaikan."
    )

    cross_label_resolution = pd.DataFrame(
        columns=[
            "review_group_id",
            "dataset",
            "label",
            "path",
            "relative_path",
            "phash",
            "resolution_action",
            "resolution_reason",
            "expert_reannotation"
        ]
    )

    excluded_cross_label_paths = set()


else:

    # --------------------------------------------------------
    # 2. AMBIL SELURUH CITRA DALAM CONFLICT COMPONENT
    #
    # Bukan hanya dua pair.
    # Seluruh connected component dikeluarkan.
    # --------------------------------------------------------

    conflict_images = (
        perceptual_candidates[
            perceptual_candidates[
                "review_group_id"
            ]
            .astype(str)
            .isin(
                conflict_review_group_ids
            )
        ]
        .copy()
        .reset_index(drop=True)
    )


    excluded_cross_label_paths = set(
        conflict_images[
            "path"
        ]
        .astype(str)
        .tolist()
    )


    if len(excluded_cross_label_paths) == 0:

        raise RuntimeError(
            "Cross-label conflict ditemukan, "
            "tetapi tidak ada path yang dapat "
            "diidentifikasi untuk resolusi."
        )


    # --------------------------------------------------------
    # 3. RESOLUTION ARTIFACT
    # --------------------------------------------------------

    cross_label_resolution = (
        conflict_images[
            [
                "review_group_id",
                "dataset",
                "label",
                "path",
                "relative_path",
                "phash"
            ]
        ]
        .copy()
    )


    cross_label_resolution[
        "resolution_action"
    ] = "exclude_from_modeling"


    cross_label_resolution[
        "resolution_reason"
    ] = (
        "ambiguous_cross_label_near_duplicate"
    )


    cross_label_resolution[
        "expert_reannotation"
    ] = False


# ------------------------------------------------------------
# 4. SIMPAN AUDIT TRAIL
# ------------------------------------------------------------

cross_label_resolution.to_csv(
    CROSS_LABEL_RESOLUTION_PATH,
    index=False
)


# ------------------------------------------------------------
# 5. BUAT MANIFEST RESOLVED
# ------------------------------------------------------------

combined_exact_deduplicated_resolved = (
    combined_exact_deduplicated[
        ~combined_exact_deduplicated[
            "path"
        ]
        .astype(str)
        .isin(
            excluded_cross_label_paths
        )
    ]
    .copy()
    .reset_index(drop=True)
)


perceptual_candidates_resolved = (
    perceptual_candidates[
        ~perceptual_candidates[
            "path"
        ]
        .astype(str)
        .isin(
            excluded_cross_label_paths
        )
    ]
    .copy()
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 6. VALIDASI
# ------------------------------------------------------------

removed_rows = (
    len(combined_exact_deduplicated)
    -
    len(combined_exact_deduplicated_resolved)
)


if removed_rows != len(
    excluded_cross_label_paths
):

    raise RuntimeError(
        "Jumlah citra yang dikeluarkan "
        "tidak konsisten."
    )


remaining_conflict_paths = (
    set(
        combined_exact_deduplicated_resolved[
            "path"
        ]
        .astype(str)
    )
    &
    excluded_cross_label_paths
)


if remaining_conflict_paths:

    raise RuntimeError(
        "Masih terdapat cross-label conflict "
        "dalam resolved manifest."
    )


# ------------------------------------------------------------
# 7. SUMMARY
# ------------------------------------------------------------

print("\nCROSS-LABEL RESOLUTION")

print(
    "Review groups resolved :",
    len(conflict_review_group_ids)
)

print(
    "Images excluded        :",
    removed_rows
)

print(
    "Before resolution      :",
    len(combined_exact_deduplicated)
)

print(
    "After resolution       :",
    len(combined_exact_deduplicated_resolved)
)


print("\nFinal dataset counts:")

display(
    combined_exact_deduplicated_resolved[
        "dataset"
    ]
    .value_counts()
    .rename_axis("dataset")
    .reset_index(name="jumlah")
)


print(
    "\nResolution artifact:"
)

print(
    CROSS_LABEL_RESOLUTION_PATH
)

Cross-label review groups: ['review_0038']

CROSS-LABEL RESOLUTION
Review groups resolved : 1
Images excluded        : 3
Before resolution      : 3519
After resolution       : 3516

Final dataset counts:


,dataset,jumlah
0,Dataset_A,3276
1,Dataset_B,240



Resolution artifact:
/content/drive/MyDrive/eksperimen_baru/baseline_v2/metrics/phash_cross_label_resolution.csv


In [ ]:
# ============================================================
# CELL 30 — MEMBUAT GROUP-AWARE MANIFEST
# AFTER CROSS-LABEL NEAR-DUPLICATE RESOLUTION
# ============================================================

# ------------------------------------------------------------
# 0. PASTIKAN CELL 29A SUDAH DIJALANKAN
# ------------------------------------------------------------

required_resolution_variables = [
    "combined_exact_deduplicated_resolved",
    "perceptual_candidates_resolved",
    "excluded_cross_label_paths"
]

missing_resolution_variables = [
    variable_name
    for variable_name
    in required_resolution_variables
    if variable_name not in globals()
]

if missing_resolution_variables:

    raise RuntimeError(
        "Output CELL 29A belum tersedia.\n"
        "Jalankan CELL 29A sebelum CELL 30.\n"
        f"Variabel hilang: {missing_resolution_variables}"
    )


# ------------------------------------------------------------
# 1. MEMULAI DARI MANIFEST YANG SUDAH DIRESOLUSI
# ------------------------------------------------------------

grouped_manifest = (
    combined_exact_deduplicated_resolved
    .copy()
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 2. VALIDASI KOLOM MANIFEST
# ------------------------------------------------------------

required_manifest_columns = {
    "path",
    "dataset",
    "label",
    "sha256",
    "phash"
}

missing_manifest_columns = (
    required_manifest_columns
    -
    set(
        grouped_manifest.columns
    )
)

if missing_manifest_columns:

    raise RuntimeError(
        "Manifest resolved belum lengkap.\n"
        f"Kolom hilang: "
        f"{sorted(missing_manifest_columns)}"
    )


if (
    grouped_manifest[
        "path"
    ]
    .duplicated()
    .any()
):

    raise RuntimeError(
        "Terdapat duplicate path "
        "sebelum group-aware manifest dibuat."
    )


# ------------------------------------------------------------
# 3. PASTIKAN CROSS-LABEL CONFLICT YANG DIRESOLUSI
#    TIDAK MASUK MANIFEST FINAL
# ------------------------------------------------------------

excluded_cross_label_paths = set(
    str(path)
    for path
    in excluded_cross_label_paths
)

manifest_path_set = set(
    grouped_manifest[
        "path"
    ]
    .astype(str)
    .tolist()
)

remaining_excluded_paths = (
    manifest_path_set
    &
    excluded_cross_label_paths
)

if remaining_excluded_paths:

    raise RuntimeError(
        "Masih terdapat citra cross-label conflict "
        "di resolved manifest:\n"
        +
        "\n".join(
            sorted(
                remaining_excluded_paths
            )
        )
    )


# ------------------------------------------------------------
# 4. NORMALISASI HASH
# ------------------------------------------------------------

grouped_manifest[
    "sha256"
] = (
    grouped_manifest[
        "sha256"
    ]
    .fillna("")
    .astype(str)
)


grouped_manifest[
    "phash"
] = (
    grouped_manifest[
        "phash"
    ]
    .fillna("")
    .astype(str)
)


if (
    grouped_manifest[
        "sha256"
    ]
    .eq("")
    .any()
):

    raise RuntimeError(
        "Terdapat SHA-256 kosong."
    )


if (
    grouped_manifest[
        "phash"
    ]
    .eq("")
    .any()
):

    raise RuntimeError(
        "Terdapat pHash kosong."
    )


# ------------------------------------------------------------
# 5. VALIDASI PERCEPTUAL CANDIDATES HASIL RESOLUSI
# ------------------------------------------------------------

if (
    not perceptual_candidates_resolved.empty
    and
    "near_duplicate_group_id"
    not in perceptual_candidates_resolved.columns
):

    raise RuntimeError(
        "Kolom near_duplicate_group_id "
        "tidak ditemukan pada "
        "perceptual_candidates_resolved."
    )


if (
    not perceptual_candidates_resolved.empty
    and
    "path"
    not in perceptual_candidates_resolved.columns
):

    raise RuntimeError(
        "Kolom path tidak ditemukan pada "
        "perceptual_candidates_resolved."
    )


resolved_candidate_path_set = set(
    perceptual_candidates_resolved[
        "path"
    ]
    .astype(str)
    .tolist()
)


# Semua kandidat resolved harus masih berada
# di resolved manifest.

candidate_paths_not_in_manifest = (
    resolved_candidate_path_set
    -
    manifest_path_set
)

if candidate_paths_not_in_manifest:

    raise RuntimeError(
        "Terdapat perceptual candidate resolved "
        "yang tidak terdapat dalam resolved manifest:\n"
        +
        "\n".join(
            sorted(
                candidate_paths_not_in_manifest
            )
        )
    )


# ------------------------------------------------------------
# 6. SAFE NEAR-DUPLICATE GROUP MAPPING
#
# Hanya kandidat same-dataset + same-label dari CELL 27
# yang memiliki near_duplicate_group_id.
#
# Cross-label conflict telah diselesaikan secara konservatif
# di CELL 29A dan tidak lagi berada pada manifest modeling.
# ------------------------------------------------------------

safe_candidate_rows = (
    perceptual_candidates_resolved[
        perceptual_candidates_resolved[
            "near_duplicate_group_id"
        ]
        .notna()
    ]
    .copy()
)


if (
    not safe_candidate_rows.empty
    and
    safe_candidate_rows[
        "path"
    ]
    .duplicated()
    .any()
):

    raise RuntimeError(
        "Satu path memiliki lebih dari "
        "satu near_duplicate_group_id."
    )


# Pastikan satu near-duplicate group
# tidak mengandung dataset berbeda.

if not safe_candidate_rows.empty:

    safe_group_dataset_check = (
        safe_candidate_rows
        .groupby(
            "near_duplicate_group_id"
        )[
            "dataset"
        ]
        .nunique()
    )

    unsafe_dataset_groups = (
        safe_group_dataset_check[
            safe_group_dataset_check > 1
        ]
    )

    if not unsafe_dataset_groups.empty:

        raise RuntimeError(
            "Terdapat safe near-duplicate group "
            "lintas dataset:\n"
            f"{unsafe_dataset_groups.to_dict()}"
        )


    # Pastikan satu safe group
    # tidak mengandung label berbeda.

    safe_group_label_check = (
        safe_candidate_rows
        .groupby(
            "near_duplicate_group_id"
        )[
            "label"
        ]
        .nunique()
    )

    unsafe_label_groups = (
        safe_group_label_check[
            safe_group_label_check > 1
        ]
    )

    if not unsafe_label_groups.empty:

        raise RuntimeError(
            "Terdapat safe near-duplicate group "
            "lintas label:\n"
            f"{unsafe_label_groups.to_dict()}"
        )


# ------------------------------------------------------------
# 7. PATH -> SAFE NEAR-DUPLICATE GROUP
# ------------------------------------------------------------

near_duplicate_mapping = (
    safe_candidate_rows
    .set_index(
        "path"
    )[
        "near_duplicate_group_id"
    ]
    .to_dict()
)


all_candidate_paths = set(
    perceptual_candidates_resolved[
        "path"
    ]
    .astype(str)
    .tolist()
)


# ------------------------------------------------------------
# 8. ATTACH GROUP INFORMATION
# ------------------------------------------------------------

grouped_manifest[
    "near_duplicate_group_id"
] = (
    grouped_manifest[
        "path"
    ]
    .astype(str)
    .map(
        near_duplicate_mapping
    )
)


grouped_manifest[
    "is_perceptual_candidate"
] = (
    grouped_manifest[
        "path"
    ]
    .astype(str)
    .isin(
        all_candidate_paths
    )
)


# ------------------------------------------------------------
# 9. MEMBUAT SPLIT GROUP ID
# ------------------------------------------------------------

def create_split_group_id(row):

    near_group = (
        row[
            "near_duplicate_group_id"
        ]
    )

    if pd.notna(
        near_group
    ):

        return (
            f"phash_{near_group}"
        )

    return (
        f"unique_{row['sha256']}"
    )


grouped_manifest[
    "split_group_id"
] = (
    grouped_manifest
    .apply(
        create_split_group_id,
        axis=1
    )
)


# ------------------------------------------------------------
# 10. GROUP SIZE
# ------------------------------------------------------------

grouped_manifest[
    "split_group_size"
] = (
    grouped_manifest
    .groupby(
        "split_group_id"
    )[
        "split_group_id"
    ]
    .transform(
        "size"
    )
)


# ------------------------------------------------------------
# 11. GROUP TYPE
# ------------------------------------------------------------

grouped_manifest[
    "split_group_type"
] = np.where(
    grouped_manifest[
        "split_group_size"
    ] > 1,
    "near_duplicate_group",
    "singleton"
)


grouped_manifest[
    "group_aware_split_required"
] = (
    grouped_manifest[
        "split_group_size"
    ] > 1
)


# ------------------------------------------------------------
# 12. FINAL SANITY CHECK
# ------------------------------------------------------------

if (
    grouped_manifest[
        "split_group_id"
    ]
    .isna()
    .any()
):

    raise RuntimeError(
        "Terdapat split_group_id kosong."
    )


if (
    grouped_manifest[
        "split_group_id"
    ]
    .astype(str)
    .str.strip()
    .eq("")
    .any()
):

    raise RuntimeError(
        "Terdapat split_group_id berupa string kosong."
    )


# Pastikan exact duplicate tidak muncul kembali.

remaining_exact_duplicates = int(
    grouped_manifest[
        "sha256"
    ]
    .duplicated(
        keep=False
    )
    .sum()
)

if remaining_exact_duplicates != 0:

    raise RuntimeError(
        "Exact duplicate kembali muncul "
        "pada grouped manifest.\n"
        f"Jumlah rows: "
        f"{remaining_exact_duplicates}"
    )


# ------------------------------------------------------------
# 13. DATASET COUNTS
# ------------------------------------------------------------

final_dataset_counts = (
    grouped_manifest[
        "dataset"
    ]
    .value_counts()
    .sort_index()
)


# ------------------------------------------------------------
# 14. SUMMARY
# ------------------------------------------------------------

print(
    "Group-aware manifest berhasil dibuat."
)


print(
    "\nCross-label conflict excluded:",
    len(
        excluded_cross_label_paths
    )
)


print(
    "\nJumlah baris manifest:",
    len(
        grouped_manifest
    )
)


print(
    "Jumlah split group:",
    grouped_manifest[
        "split_group_id"
    ]
    .nunique()
)


print(
    "Jumlah kandidat near-duplicate tersisa:",
    int(
        grouped_manifest[
            "is_perceptual_candidate"
        ]
        .sum()
    )
)


print(
    "Jumlah citra dalam group-aware split:",
    int(
        grouped_manifest[
            "group_aware_split_required"
        ]
        .sum()
    )
)


print(
    "Jumlah safe near-duplicate groups:",
    grouped_manifest.loc[
        grouped_manifest[
            "group_aware_split_required"
        ],
        "split_group_id"
    ]
    .nunique()
)


print(
    "Exact duplicate tersisa:",
    remaining_exact_duplicates
)


print(
    "\nFinal dataset counts:"
)

display(
    final_dataset_counts
    .rename_axis(
        "dataset"
    )
    .reset_index(
        name="jumlah"
    )
)


# ------------------------------------------------------------
# 15. PREVIEW SAFE GROUPS
# ------------------------------------------------------------

display(
    grouped_manifest.loc[
        grouped_manifest[
            "group_aware_split_required"
        ],
        [
            "dataset",
            "label",
            "relative_path",
            "phash",
            "near_duplicate_group_id",
            "split_group_id",
            "split_group_size",
            "split_group_type"
        ]
    ]
    .head(30)
)

Group-aware manifest berhasil dibuat.

Cross-label conflict excluded: 3

Jumlah baris manifest: 3516
Jumlah split group: 2989
Jumlah kandidat near-duplicate tersisa: 811
Jumlah citra dalam group-aware split: 811
Jumlah safe near-duplicate groups: 284
Exact duplicate tersisa: 0

Final dataset counts:


,dataset,jumlah
0,Dataset_A,3276
1,Dataset_B,240


,dataset,label,relative_path,phash,near_duplicate_group_id,split_group_id,split_group_size,split_group_type
5,Dataset_A,blast,blast/101016.jpg,b0ccb3dcb0d98536,near_dup_0001,phash_near_dup_0001,6,near_duplicate_group
8,Dataset_A,bacterial_leaf_blight,bacterial_leaf_blight/100582.jpg,c7b218fca7dd02e0,near_dup_0002,phash_near_dup_0002,2,near_duplicate_group
11,Dataset_A,tungro,tungro/103259.jpg,e29f03e816edc728,near_dup_0003,phash_near_dup_0003,2,near_duplicate_group
19,Dataset_A,blast,blast/103036.jpg,a0d7e482f58cf08f,near_dup_0004,phash_near_dup_0004,8,near_duplicate_group
27,Dataset_A,blast,blast/107489.jpg,d8833f438fb265c4,near_dup_0005,phash_near_dup_0005,8,near_duplicate_group
40,Dataset_A,tungro,tungro/102907.jpg,ace8f1d1c50b84b7,near_dup_0006,phash_near_dup_0006,2,near_duplicate_group
43,Dataset_A,blast,blast/102739.jpg,e781d4f5d68580ec,near_dup_0007,phash_near_dup_0007,2,near_duplicate_group
45,Dataset_A,bacterial_leaf_blight,bacterial_leaf_blight/104018.jpg,c68618f4bfcd07e0,near_dup_0008,phash_near_dup_0008,2,near_duplicate_group
53,Dataset_A,tungro,tungro/110228.jpg,99e436cb57a1503e,near_dup_0009,phash_near_dup_0009,4,near_duplicate_group
54,Dataset_A,tungro,tungro/107449.jpg,ced2a5c816a3f195,near_dup_0010,phash_near_dup_0010,3,near_duplicate_group


In [ ]:
# ============================================================
# CELL 31 — VALIDASI GROUP-AWARE MANIFEST
# ============================================================

required_columns = [
    "path",
    "dataset",
    "label",
    "sha256",
    "phash",
    "split_group_id",
    "split_group_size",
    "split_group_type"
]

missing_columns = [
    column
    for column in required_columns
    if column not in grouped_manifest.columns
]

if missing_columns:
    raise ValueError(
        f"Kolom berikut belum tersedia: {missing_columns}"
    )

group_validation = (
    grouped_manifest
    .groupby("split_group_id")
    .agg(
        group_size=(
            "path",
            "size"
        ),
        dataset_count=(
            "dataset",
            "nunique"
        ),
        label_count=(
            "label",
            "nunique"
        ),
        path_count=(
            "path",
            "nunique"
        )
    )
    .reset_index()
)

cross_dataset_groups = (
    group_validation[
        group_validation[
            "dataset_count"
        ] > 1
    ]
)

cross_label_groups = (
    group_validation[
        group_validation[
            "label_count"
        ] > 1
    ]
)

duplicate_path_groups = (
    group_validation[
        group_validation[
            "path_count"
        ]
        != group_validation[
            "group_size"
        ]
    ]
)

missing_group_ids = int(
    grouped_manifest[
        "split_group_id"
    ].isna().sum()
)

empty_group_ids = int(
    (
        grouped_manifest[
            "split_group_id"
        ]
        .astype(str)
        .str.strip()
        == ""
    ).sum()
)

print(
    "Group lintas dataset :",
    len(cross_dataset_groups)
)

print(
    "Group lintas label   :",
    len(cross_label_groups)
)

print(
    "Group path duplikat  :",
    len(duplicate_path_groups)
)

print(
    "Group ID kosong      :",
    missing_group_ids + empty_group_ids
)

if len(cross_dataset_groups) > 0:
    display(cross_dataset_groups)

    raise ValueError(
        "Terdapat split group yang melintasi dataset."
    )

if len(cross_label_groups) > 0:
    display(cross_label_groups)

    raise ValueError(
        "Terdapat split group yang memiliki lebih dari satu label."
    )

if len(duplicate_path_groups) > 0:
    display(duplicate_path_groups)

    raise ValueError(
        "Terdapat path duplikat dalam split group."
    )

if missing_group_ids + empty_group_ids > 0:
    raise ValueError(
        "Terdapat split_group_id kosong."
    )

print(
    "\nValidasi group-aware manifest: LULUS"
)

Group lintas dataset : 0
Group lintas label   : 0
Group path duplikat  : 0
Group ID kosong      : 0

Validasi group-aware manifest: LULUS


In [ ]:
# ============================================================
# CELL 32 — MENYIMPAN GROUP-AWARE MANIFEST
# ============================================================

dataset_a_grouped_manifest = (
    grouped_manifest[
        grouped_manifest[
            "dataset"
        ] == "Dataset_A"
    ]
    .copy()
    .reset_index(drop=True)
)

dataset_b_grouped_manifest = (
    grouped_manifest[
        grouped_manifest[
            "dataset"
        ] == "Dataset_B"
    ]
    .copy()
    .reset_index(drop=True)
)

DATASET_A_GROUPED_PATH = (
    MANIFEST_DIR
    / "dataset_a_grouped_manifest.csv"
)

DATASET_B_GROUPED_PATH = (
    MANIFEST_DIR
    / "dataset_b_grouped_manifest.csv"
)

COMBINED_GROUPED_PATH = (
    MANIFEST_DIR
    / "combined_grouped_manifest.csv"
)

GROUP_VALIDATION_PATH = (
    METRICS_DIR
    / "split_group_validation.csv"
)

dataset_a_grouped_manifest.to_csv(
    DATASET_A_GROUPED_PATH,
    index=False
)

dataset_b_grouped_manifest.to_csv(
    DATASET_B_GROUPED_PATH,
    index=False
)

grouped_manifest.to_csv(
    COMBINED_GROUPED_PATH,
    index=False
)

group_validation.to_csv(
    GROUP_VALIDATION_PATH,
    index=False
)

print(
    "Group-aware manifest berhasil disimpan:"
)

print("-", DATASET_A_GROUPED_PATH)
print("-", DATASET_B_GROUPED_PATH)
print("-", COMBINED_GROUPED_PATH)
print("-", GROUP_VALIDATION_PATH)

print("\nJumlah akhir:")
print(
    "Dataset A:",
    len(dataset_a_grouped_manifest)
)

print(
    "Dataset B:",
    len(dataset_b_grouped_manifest)
)

print(
    "Gabungan :",
    len(grouped_manifest)
)

Group-aware manifest berhasil disimpan:
- /content/drive/MyDrive/eksperimen_baru/baseline_v2/manifests/dataset_a_grouped_manifest.csv
- /content/drive/MyDrive/eksperimen_baru/baseline_v2/manifests/dataset_b_grouped_manifest.csv
- /content/drive/MyDrive/eksperimen_baru/baseline_v2/manifests/combined_grouped_manifest.csv
- /content/drive/MyDrive/eksperimen_baru/baseline_v2/metrics/split_group_validation.csv

Jumlah akhir:
Dataset A: 3276
Dataset B: 240
Gabungan : 3516
